# Hmall 협력사 문서 RAG 

**병합 범위는 1단계(수집·Ingestion)와 2단계(골든 데이터셋)까지.**
이후 RAG 워크플로우·성능 평가·챗봇 단계는 이 노트북에서 제외했다.

### 파이프라인

| 단계 | 내용 | 핵심 선택 |
|---|---|---|
| 1 | 데이터 수집·Ingestion | 로더 5종 × 문서 8종 벤치마크 → 문서별 최적 로더 판단. 로더 라우팅 + 청킹 + 벡터스토어 |
| 2 | 골든 데이터셋 생성·큐레이션 | 22문항 / 2단계 자동 검수 / 무답변 문항 자동 검증 |
| 3 | RAG 검색·생성 워크플로우 | Chroma + LCEL, 표 행 조건 준수 프롬프트 |
| 4 | 성능 평가 | Ragas 표준 4지표 + 커스텀 2지표 |
| 5 | 챗봇 | Streamlit (`app.py`) |

### 대상 문서 8종

| # | 문서 | 페이지 |
|---|---|---:|
| 1 | 협력사 운영 통합 안내서 | 23 |
| 2 | Hmall 소개서 | 30 |
| 3 | 쇼라 소개서 | 22 |
| 4 | 광고 상품 소개서 | 24 |
| 5 | 협력사시스템 사용법 | 10 |
| 6 | 신규 협력사 입점 절차 안내 | 21 |
| 7 | 데이터 영역 광고 제안서 | 8 |
| 8 | **QA가이드** (신규 추가, 실제 38p) | 38 |

총 176페이지. 참조 경로는 `./files/`, 골든 데이터셋은 `golden_dataset.json`.


In [1]:
# =====================================================================
# 0. 공통 설정
# =====================================================================
# 이후 모든 단계가 이 셀의 경로·모델 설정을 공유한다.
import os
from pathlib import Path

from dotenv import load_dotenv, find_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_core.callbacks import UsageMetadataCallbackHandler

# .env 파일을 찾아 환경변수로 읽어들인다.
#   usecwd=True  -> 현재 작업 디렉터리부터 위로 올라가며 .env 를 찾는다.
#   override=True -> 이미 셸에 같은 이름의 변수가 있어도 .env 값으로 덮어쓴다.
#                    (셸에 남은 옛 키 때문에 엉뚱한 모델을 부르는 사고를 막는다)
load_dotenv(find_dotenv(".env", usecwd=True), override=True)

MODEL     = os.environ["OPENAI_DEFAULT_MODEL"]    # 파싱·생성용 상위 모델
LOWER     = os.environ["OPENAI_LOWER_MODEL"]      # 단순 페이지 파싱용 경량 모델
JUDGE     = os.environ["OPENAI_JUDGE_MODEL"]      # 검수 심사위원
EMBEDDING = os.environ["OPENAI_EMBEDDING_MODEL"]

# 1M 토큰당 (입력, 출력) USD — 로더 비용 비교에 사용
# VLM 파싱 비용을 실제 토큰 사용량 × 단가로 환산할 때 쓴다.
# 임베딩은 출력 토큰이 없으므로 출력 단가가 0이다.
PRICES = {MODEL: (2.50, 15.00), LOWER: (0.75, 4.50), EMBEDDING: (0.02, 0.0)}

PDF_DIR         = Path("./files")        # 대상 PDF 8종
CACHE_DIR       = Path("./md_cache")     # VLM 파싱 캐시 (재실행 시 API 비용 0)
CHROMA_PATH     = "./chroma_db"
COLLECTION_NAME = "langchain"
GOLDEN_PATH     = "golden_dataset.json"
BENCH_CSV       = "loader_benchmark.csv"

# 캐시 폴더가 없으면 만든다. exist_ok=True 라 이미 있어도 에러가 나지 않는다.
CACHE_DIR.mkdir(exist_ok=True)

# 대상 PDF 목록. sorted 로 파일명 순서를 고정해야
# 실행할 때마다 문서 순서가 바뀌지 않아 결과를 비교할 수 있다.
PDF_LIST = sorted(PDF_DIR.glob("*.pdf"))

usage = UsageMetadataCallbackHandler()   # 토큰 사용량 집계용

# 임베딩 객체는 여기서 한 번만 만들어 1·2단계가 공유한다.
# 청킹·벡터스토어·골든셋이 서로 다른 임베딩을 쓰면 벡터 공간이 달라져
# 검색 자체가 성립하지 않는다.
embeddings = OpenAIEmbeddings(model=EMBEDDING)

print(f"MODEL={MODEL} / LOWER={LOWER} / JUDGE={JUDGE} / EMBEDDING={EMBEDDING}")
print(f"대상 PDF {len(PDF_LIST)}개:")
for p in PDF_LIST:
    print("  -", p.name)

MODEL=gpt-5.4 / LOWER=gpt-5.4-mini / JUDGE=gpt-5.5 / EMBEDDING=text-embedding-3-small
대상 PDF 8개:
  - 1. 협력사 운영 통합 안내서 (p.23).pdf
  - 2. Hmall 소개서 (p.30).pdf
  - 3. 쇼라 소개서 (p.22).pdf
  - 4. 광고 상품 소개서 (p.24).pdf
  - 5. 협력사시스템 사용법 (p.10).pdf
  - 6. 신규 협력사 입점 절차 안내 (p.21).pdf
  - 7. 데이터 영역 광고 제안서 (p.8).pdf
  - 8. QA가이드 (p.38).pdf


---
# 1단계-A. PDF 로더 5종 비교

문서마다 성격이 달라서 하나의 로더로 8개를 다 처리할 수 없음.

어떤 문서는 로컬 로더로 충분하고, 어떤 문서는 로컬 로더로 텍스트 추출 자체가 안되어서 로컬 로더로 불가능.

따라서 각 pdf 문서별 loader 성능을 측정하고 pdf 문서마다 적합한 loader를 선택한다.

### 비교 대상

| 로더 | 방식 | 비용 |
|---|---|---|
| PyPDF | 텍스트 레이어 추출 | 0 |
| PDFPlumber | 텍스트 레이어 + 표 격자 분석 | 0 |
| PyMuPDF | 텍스트 레이어 추출 (C 기반, 빠름) | 0 |
| PyMuPDF+Markdown | 위 + 표를 마크다운으로 변환 | 0 |
| **하이브리드 VLM** | 페이지 이미지 + 텍스트 레이어를 함께 비전 모델에 투입 | **API 과금** |

### 측정 지표

- **전체문자수** — 회수한 글자 수. 단, 이 값만 보면 오판한다(아래 참고).
- **순수문자수** — `**`, `~~`, `<br>`, `|` 같은 마크업을 제거한 실제 내용 글자 수.
- **마크다운표수** — 만들어낸 표 개수.
- **마크업비율** — 기호만 늘어난 가짜 문자 수 (클수록 의미없는 기호가 많이 포함되어있음)
- **행중복률** — 한 표 안에서 동일한 첫 셀이 반복되는 비율. 열이 붕괴하면 올라간다.
- **비용(USD)** — 로컬은 0. VLM은 실제 토큰 사용량 × 단가.

**전체문자수를 그대로 믿으면 안 되는 이유:** 표 변환에 실패한 로더는
`~~**현**~~**대**<br>~~**백화점**~~` 처럼 마크업만 잔뜩 return.
내용은 오히려 깨졌는데 문자수는 커지므로 순수문자수를 함께 본다.


In [2]:
# =====================================================================
# 1-A단계: PDF 로더 5종 비교 벤치마크
# ---------------------------------------------------------------------
# 8개 문서를 5종 로더로 각각 파싱해 "문서별로 어떤 로더가 최적인가"를 측정한다.
#   PyPDF / PDFPlumber / PyMuPDF / PyMuPDF+Markdown  : 로컬, 비용 0
#   하이브리드 VLM (페이지 이미지 + 텍스트 레이어)      : API 비용 발생
#
# 판단 기준
#   - 전체문자수   : 내용을 얼마나 회수했는가 (0이면 그 로더로는 못 읽는 문서)
#   - 마크다운표수 : 표 구조를 살렸는가
#   - 비용(USD)    : 로컬은 0, VLM은 실제 토큰 사용량 기준
# 로컬 로더로 충분한 문서에 VLM을 쓰면 돈만 버리므로, 문서별로 나눠 쓰는 것이 목표다.
# =====================================================================
import os, re, time, json, base64, hashlib, math
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pymupdf
import pandas as pd
from dotenv import load_dotenv, find_dotenv
from openai import OpenAI
from langchain_community.document_loaders import (
    PyPDFLoader, PDFPlumberLoader, PyMuPDFLoader,
)

client = OpenAI()

USAGE_DB  = CACHE_DIR / "usage.json"
DPI = 200

# ---------------------------------------------------------------------
# VLM 재실측 플래그
# ---------------------------------------------------------------------
#
#   REMEASURE_MISSING_ONLY = True  -> 실측 기록이 없는 페이지만 재호출
#                                     이미 실측된 페이지는 건너뛰므로 이쪽을 권장
#   FORCE_REMEASURE        = True  -> 캐시를 무시하고 전 페이지 재호출
#   둘 다 False             -> 캐시 그대로 사용 (비용 0, 기본값)
#
# ---------------------------------------------------------------------
FORCE_REMEASURE        = False
REMEASURE_MISSING_ONLY = False

PDF_LIST = sorted(PDF_DIR.glob("*.pdf"))

MARKUP_RE = re.compile(r"(\*\*|~~|<br>|_|\||[-:]{2,})")


# 마크다운 개수
def count_markdown_tables(text):
    pattern = r'\|.*\|\s*\n\|(?:\s*:?-+:?\s*\|)+'
    return len(re.findall(pattern, text))


# 마크업 기호를 지운 '내용' 글자 수를 센다.
def clean_chars(text: str) -> int:
    """마크업 기호를 뺀 '내용' 글자 수.

    표를 못 만든 로더가 `~~**현**~~**대**<br>` 처럼 마크업만 잔뜩 뱉으면
    전체문자수가 부풀어 오른다. 마크업을 제거해야 실제 회수량이 보인다.
    """
    return len(re.sub(r"\s+", "", MARKUP_RE.sub("", text or "")))


# 표의 헤더 열 수와 데이터 행 열 수가 맞는 비율.
# 실측해 보니 모든 로더가 1.0 이 나와 변별력이 없었다.
# (PyMuPDF+Markdown 은 열이 모자라면 Col3, Col6 같은 이름을 자동으로 채운다)
# 그래서 조회 화면에서는 빼고 CSV 에만 남겨 뒀다.
def table_consistency(text: str) -> float:
    """마크다운 표의 (헤더 열 수 == 데이터 행 열 수) 비율.

    표는 만들었지만 열이 어긋나 있으면 행/열 대응이 깨진 것이므로
    표 개수만으로는 품질을 알 수 없다. 1.0에 가까울수록 온전한 표다.
    """
    lines = (text or "").splitlines()
    total = good = 0
    header_cols = None
    for line in lines:
        st = line.strip()
        if not st.startswith("|"):
            header_cols = None
            continue
        ncol = st.count("|") - 1
        if set(st) <= set("|-: "):          # 구분선
            header_cols = ncol
            continue
        if header_cols is not None:          # 구분선 이후의 데이터 행
            total += 1
            good += (ncol == header_cols)
    return round(good / total, 3) if total else None


# 표의 첫 열에 같은 값이 반복되는 비율.
# 주의: 높다고 무조건 나쁜 것이 아니다. VLM 은 세로 병합 셀을 각 행에
# 반복해 채우도록 지시받았으므로 정상 동작에서도 값이 올라간다.
# 마크업비율까지 함께 높을 때만 '열 붕괴'로 해석한다.
def cell_dup_ratio(text: str) -> float:
    """같은 표 안에서 완전히 동일한 데이터 행이 차지하는 비율.

    한 열의 내용이 모든 행에 통째로 복사되는 붕괴가 일어나면 값이 올라간다.
    낮을수록 좋다.
    """
    rows = [l.strip() for l in (text or "").splitlines()
            if l.strip().startswith("|") and not set(l.strip()) <= set("|-: ")]
    if len(rows) < 2:
        return None
    first_cells = [r.split("|")[1].strip() for r in rows if len(r.split("|")) > 2]
    if not first_cells:
        return None
    from collections import Counter
    most = Counter(first_cells).most_common(1)[0][1]
    return round(most / len(first_cells), 3)


# ---------------------------------------------------------------------
# 하이브리드 VLM 파싱
# ---------------------------------------------------------------------
PROMPT = """아래는 같은 PDF 페이지의 두 가지 정보다.
(A) 페이지 이미지 - 표의 행/열 구조와 화면 배치를 여기서 읽는다.
(B) PDF 내부 텍스트 레이어 - 글자 표기(철자)가 정확하다. 단 순서가 뒤섞여 구조 정보는 없다.

규칙:
1. 구조는 (A)에서, 글자 표기는 (B)에서 가져온다. (B)에 있는 단어는 (B)의 철자를 그대로 쓰고
   이미지로 읽은 추측 철자로 덮어쓰지 않는다. ('탭/법', '뎁스/탑스', '日/타' 등 혼동 금지)
2. 표가 있으면 마크다운 파이프 테이블로 출력한다.
   - 열 개수는 데이터 행을 기준으로 정한다. 머리글이 없는 열(하위 구분 등)이 있으면
     헤더에 이름을 새로 지어 붙여서 열을 늘린다. 값을 지워 열 수를 맞추는 것은 절대 금지.
   - 세로로 병합된 셀은 이미지에서 그 셀이 세로로 덮고 있는 행 범위를 정확히 확인해서
     해당하는 모든 행에 값을 반복해 채운다. 그룹 경계를 틀리지 마라.
3. 표 아래에 `#### 표 행 풀어쓰기` 를 붙이고, 각 데이터 행을 한 문장씩 쓴다.
   모든 열의 값이 문장 안에 빠짐없이 들어가야 하고, 데이터 행 수와 문장 수가 같아야 한다.
4. 제목은 #, 소제목은 ##. 화면 캡처/차트/흐름도 안의 글자도 모두 옮기고,
   절차 흐름도는 "1단계 -> 2단계" 형태로 순서를 풀어 쓴다.
5. 금액·기간·수수료율·URL·날짜는 원문 표기 그대로 옮긴다.
6. 해설/요약/추측 금지. 페이지에 실제로 적힌 내용만 출력한다.

=== (B) 텍스트 레이어 ===
{layer}
=== 끝 ==="""


# 페이지 난이도에 따라 모델을 고른다.
# 구조 복원이 어려운 페이지(표 있음 / 스캔형)만 비싼 상위 모델로 보내고
# 나머지는 경량 모델로 처리해 비용을 줄인다.
def pick_model(page, layer: str) -> str:
    """표가 있거나 스캔형 페이지는 구조 복원이 어려우므로 상위 모델."""
    if len(layer.strip()) < 100:
        return MODEL
    try:
        if len(page.find_tables().tables) > 0:
            return MODEL
    except Exception:
        pass
    return LOWER


# 이미지 1장이 몇 토큰으로 계산되는지 추정한다(실측 기록이 없는 캐시 페이지용).
# OpenAI 는 이미지를 512px 타일로 쪼개 과금하므로 그 규칙을 따라 계산한다.
# 실측과 비교해 보니 이 추정치는 실제보다 낮게 나오는 경향이 있다.
def _image_tokens(w_px: int, h_px: int) -> int:
    """OpenAI high-detail 이미지 토큰 추정식 (512px 타일 기준)."""
    w, h = float(w_px), float(h_px)
    if max(w, h) > 2048:
        s = 2048 / max(w, h); w, h = w * s, h * s
    if min(w, h) > 768:
        s = 768 / min(w, h); w, h = w * s, h * s
    return 85 + 170 * (math.ceil(w / 512) * math.ceil(h / 512))


def _load_usage_db():
    if USAGE_DB.exists():
        try:
            return json.loads(USAGE_DB.read_text(encoding="utf-8"))
        except Exception:
            pass
    return {}


_usage_db = _load_usage_db()


def vlm_parse_page(pdf_path, page_no):
    """페이지 1장 -> (마크다운, 모델, 입력토큰, 출력토큰, 실측여부)"""
    with pymupdf.open(pdf_path) as doc:
        page = doc[page_no]
        layer = page.get_text("text")
        model = pick_model(page, layer)
        pix = page.get_pixmap(dpi=DPI)
        png, w, h = pix.tobytes("png"), pix.width, pix.height

    # 캐시 키에 파일명·페이지·모델·DPI·프롬프트 버전(v2)을 모두 넣는다.
    # 모델 라우팅이나 DPI 를 바꾸면 키가 달라져 해당 페이지만 자동으로 다시 파싱되고,
    # 프롬프트를 고칠 때는 v2 를 올려 전체 캐시를 무효화할 수 있다.
    key = hashlib.md5(
        f"{os.path.basename(pdf_path)}|{page_no}|{model}|{DPI}|v2".encode()
    ).hexdigest()[:16]
    cache_file = CACHE_DIR / f"{key}.md"
    prompt_text = PROMPT.format(
        layer=layer if layer.strip() else "(텍스트 레이어 없음 - 이미지만으로 판독)")

    rec = _usage_db.get(key)
    # 캐시를 쓸 수 있는가? 위 재실측 플래그가 이 판단을 뒤집는다.
    use_cache = cache_file.exists() and not FORCE_REMEASURE
    if use_cache and REMEASURE_MISSING_ONLY and rec is None:
        use_cache = False                          # 실측 기록이 없는 페이지만 다시 부른다

    if use_cache:
        md = cache_file.read_text(encoding="utf-8")
        if rec:                                    # 실측 사용량이 남아 있는 경우
            return md, model, rec["in"], rec["out"], True
        # 캐시만 있고 사용량 기록이 없으면 추정한다
        import tiktoken
        enc = tiktoken.get_encoding("o200k_base")
        tin = _image_tokens(w, h) + len(enc.encode(prompt_text))
        tout = len(enc.encode(md))
        return md, model, tin, tout, False

    resp = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": [
            {"type": "text", "text": prompt_text},
            {"type": "image_url", "image_url": {
                "url": f"data:image/png;base64,{base64.b64encode(png).decode()}",
                "detail": "high"}},
        ]}],
    )
    md = (resp.choices[0].message.content or "").strip()
    cache_file.write_text(md, encoding="utf-8")
    tin, tout = resp.usage.prompt_tokens, resp.usage.completion_tokens
    _usage_db[key] = {"in": tin, "out": tout, "model": model}
    return md, model, tin, tout, True


def vlm_load(pdf_path):
    """PDF 전체를 하이브리드 VLM으로 파싱."""
    with pymupdf.open(pdf_path) as d:
        n = d.page_count
    with ThreadPoolExecutor(max_workers=8) as ex:
        out = list(ex.map(lambda i: vlm_parse_page(pdf_path, i), range(n)))

    text = "\n\n".join(r[0] for r in out)
    # 페이지마다 쓰인 모델이 다르므로 각자의 단가를 적용해 합산한다.
    cost = sum(
        tin / 1e6 * PRICES[m][0] + tout / 1e6 * PRICES[m][1]
        for _, m, tin, tout, _ in out
    )
    measured = sum(1 for r in out if r[4])
    return text, n, cost, measured


# ---------------------------------------------------------------------
# 벤치마크
# ---------------------------------------------------------------------
LOADERS = [
    ("PyPDF",            lambda p: PyPDFLoader(str(p)).load()),
    ("PDFPlumber",       lambda p: PDFPlumberLoader(str(p)).load()),
    ("PyMuPDF",          lambda p: PyMuPDFLoader(str(p), mode="page").load()),
    ("PyMuPDF+Markdown", lambda p: PyMuPDFLoader(str(p), mode="page",
                                                 extract_tables="markdown").load()),
]


def benchmark(pdf_path):
    rows = []
    name = pdf_path.name

    for loader_name, fn in LOADERS:
        t0 = time.perf_counter()
        try:
            docs = fn(pdf_path)
            text = "\n".join(d.page_content for d in docs)
            rows.append({
                "문서": name, "로더": loader_name,
                "페이지": len(docs),
                "전체문자수": len(text),
                "순수문자수": clean_chars(text),
                "마크다운표수": count_markdown_tables(text),
                "표정합성": table_consistency(text),
                "행중복률": cell_dup_ratio(text),
                "실행시간(s)": round(time.perf_counter() - t0, 2),
                "비용(USD)": 0.0,
                "상태": "OK",
            })
        except Exception as e:
            rows.append({"문서": name, "로더": loader_name, "페이지": None,
                         "전체문자수": None, "순수문자수": None, "마크다운표수": None,
                         "표정합성": None, "행중복률": None,
                         "실행시간(s)": None, "비용(USD)": None,
                         "상태": f"ERROR: {type(e).__name__}"})

    t0 = time.perf_counter()
    try:
        text, n, cost, measured = vlm_load(pdf_path)
        rows.append({
            "문서": name, "로더": "하이브리드 VLM",
            "페이지": n,
            "전체문자수": len(text),
            "순수문자수": clean_chars(text),
            "마크다운표수": count_markdown_tables(text),
            "표정합성": table_consistency(text),
            "행중복률": cell_dup_ratio(text),
            "실행시간(s)": round(time.perf_counter() - t0, 2),
            "비용(USD)": round(cost, 4),
            "상태": f"OK (실측 {measured}/{n}p)",
        })
    except Exception as e:
        rows.append({"문서": name, "로더": "하이브리드 VLM", "페이지": None,
                     "전체문자수": None, "순수문자수": None, "마크다운표수": None,
                     "표정합성": None, "행중복률": None,
                     "실행시간(s)": None, "비용(USD)": None,
                     "상태": f"ERROR: {type(e).__name__}: {e}"})
    return rows


# ---------- 8개 문서 × 5개 로더 실행 ----------
if FORCE_REMEASURE:
    print("⚠️ FORCE_REMEASURE=True — 캐시를 무시하고 전 페이지를 다시 호출합니다 (약 $3.95)\n")
elif REMEASURE_MISSING_ONLY:
    print("⚠️ REMEASURE_MISSING_ONLY=True — 실측 기록이 없는 페이지만 다시 호출합니다 (약 $2.96)\n")
else:
    print("캐시 사용 모드 (API 비용 0). 실측 토큰이 필요하면 위 플래그를 켜세요.\n")

all_rows = []
for p in PDF_LIST:
    print(f"▶ {p.name}")
    all_rows += benchmark(p)
    USAGE_DB.write_text(json.dumps(_usage_db, ensure_ascii=False), encoding="utf-8")

pd.DataFrame(all_rows).to_csv(BENCH_CSV, index=False, encoding="utf-8-sig")
print(f"\n저장: {BENCH_CSV}")

/home/ubuntu/work/kosa-ict-genai-2026-hyundai/src/exercise/JieunKwonnn/mini_project01/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


캐시 사용 모드 (API 비용 0). 실측 토큰이 필요하면 위 플래그를 켜세요.

▶ 1. 협력사 운영 통합 안내서 (p.23).pdf
▶ 2. Hmall 소개서 (p.30).pdf
▶ 3. 쇼라 소개서 (p.22).pdf
▶ 4. 광고 상품 소개서 (p.24).pdf
▶ 5. 협력사시스템 사용법 (p.10).pdf
▶ 6. 신규 협력사 입점 절차 안내 (p.21).pdf
▶ 7. 데이터 영역 광고 제안서 (p.8).pdf
▶ 8. QA가이드 (p.38).pdf

저장: loader_benchmark.csv


In [3]:
# =====================================================================
# 저장된 로더 벤치마크 결과 조회 — 문서별 표 8개
# ---------------------------------------------------------------------
# 이미 `loader_benchmark.csv`가 있다면 이 셀만 실행해서 바로 결과를 본다.
#
# 지표 읽는 법
#   마크다운표수 : 클수록 좋다. 단, 깨진 표도 개수에는 잡힌다.
#   마크업비율   : 클수록 나쁘다. 내용이 아니라 **, ~~, <br>, | 같은 기호가 늘어난 것.
#                  공백도 함께 세므로 절대값이 아니라 같은 문서 안에서 로더끼리 비교한다.
#   행중복률     : 단독으로는 좋고 나쁨을 판정할 수 없다.
#                  VLM은 세로 병합 셀을 각 행에 반복 채우도록 프롬프트가 지시하므로
#                  정상 동작인데도 값이 올라간다(4번 문서 0.33).
#                  마크업비율까지 함께 높을 때만 열 붕괴로 본다(2번 문서 0.578).
# =====================================================================
import pandas as pd
from pathlib import Path

pd.set_option("display.width", 200)

# 표에서 로더를 항상 이 순서로 보여준다(가벼운 로더 -> 무거운 로더).
# 문서마다 순서가 바뀌면 여러 표를 나란히 비교할 수 없다.
LOADER_ORDER = ["PyPDF", "PDFPlumber", "PyMuPDF", "PyMuPDF+Markdown", "하이브리드 VLM"]
# 표에 실을 열과 그 순서
COLS = ["전체문자수", "순수문자수", "마크업비율", "마크다운표수",
        "행중복률", "실행시간(s)", "비용(USD)", "상태"]

if not Path(BENCH_CSV).exists():
    print(f"⚠️ {BENCH_CSV} 가 없습니다. 위 1-A 벤치마크 셀을 먼저 실행하세요.")
else:
    bench = pd.read_csv(BENCH_CSV)
    # 마크업비율 = 전체 글자 중 기호(**, ~~, <br>, |)와 공백이 차지하는 비율.
    # CSV에는 없는 파생 지표라 여기서 계산한다.
    bench["마크업비율"] = (1 - bench["순수문자수"] / bench["전체문자수"]).round(3)
    # 파일명 앞의 숫자를 문서번호로 뽑아 정렬·그룹핑 키로 쓴다.
    bench["문서번호"] = bench["문서"].str.extract(r"^(\d+)").astype(int)
    # 로더 열을 '순서가 있는 범주형'으로 바꾼다.
    # 이렇게 해야 아래 sort_values("로더")가 가나다순이 아니라
    # LOADER_ORDER 순서로 정렬된다.
    bench["로더"] = pd.Categorical(bench["로더"], categories=LOADER_ORDER, ordered=True)

    # 문서번호로 묶어 문서당 표 하나씩(총 8개) 출력한다.
    for num, grp in bench.groupby("문서번호"):
        doc_name = grp["문서"].iloc[0]
        # 로더가 실패하면 페이지가 NaN 이므로 dropna 후 첫 값을 쓴다.
        pages = grp["페이지"].dropna()
        page_txt = f"{int(pages.iloc[0])}p" if len(pages) else "?p"

        print("=" * 100)
        print(f"[{num}] {doc_name}   ({page_txt})")
        print("=" * 100)

        table = (grp.sort_values("로더")
                    .set_index("로더")[COLS])
        display(table)

[1] 1. 협력사 운영 통합 안내서 (p.23).pdf   (23p)


,전체문자수,순수문자수,마크업비율,마크다운표수,행중복률,실행시간(s),비용(USD),상태
로더,,,,,,,,
PyPDF,8388,7267,0.134,0,NaN,3.42,0.0000,OK
PDFPlumber,8357,7267,0.130,0,NaN,5.41,0.0000,OK
PyMuPDF,8246,7267,0.119,0,NaN,0.24,0.0000,OK
PyMuPDF+Markdown,14316,10826,0.244,21,0.058,2.06,0.0000,OK
하이브리드 VLM,21422,15882,0.259,24,0.052,10.19,0.3298,OK (실측 23/23p)


[2] 2. Hmall 소개서 (p.30).pdf   (30p)


,전체문자수,순수문자수,마크업비율,마크다운표수,행중복률,실행시간(s),비용(USD),상태
로더,,,,,,,,
PyPDF,7266,5434,0.252,0,NaN,3.75,0.0000,OK
PDFPlumber,6812,5434,0.202,0,NaN,8.39,0.0000,OK
PyMuPDF,6235,5434,0.128,0,NaN,0.16,0.0000,OK
PyMuPDF+Markdown,12335,6769,0.451,11,0.578,3.72,0.0000,OK
하이브리드 VLM,19709,13970,0.291,25,0.090,5.47,0.2396,OK (실측 30/30p)


[3] 3. 쇼라 소개서 (p.22).pdf   (22p)


...

,전체문자수,순수문자수,마크업비율,마크다운표수,행중복률,실행시간(s),비용(USD),상태
로더,,,,,,,,
PyPDF,6598,5731,0.131,0,NaN,12.82,0.0000,OK
PDFPlumber,6598,5731,0.131,0,NaN,36.11,0.0000,OK
PyMuPDF,6493,5727,0.118,0,NaN,0.40,0.0000,OK
PyMuPDF+Markdown,7006,5960,0.149,2,0.091,6.64,0.0000,OK
하이브리드 VLM,17413,12201,0.299,20,0.045,13.20,0.1232,OK (실측 21/21p)


[7] 7. 데이터 영역 광고 제안서 (p.8).pdf   (8p)


,전체문자수,순수문자수,마크업비율,마크다운표수,행중복률,실행시간(s),비용(USD),상태
로더,,,,,,,,
PyPDF,1096,829,0.244,0,NaN,0.20,0.0000,OK
PDFPlumber,1115,829,0.257,0,NaN,0.44,0.0000,OK
PyMuPDF,1001,829,0.172,0,NaN,0.03,0.0000,OK
PyMuPDF+Markdown,1755,1299,0.260,4,0.120,0.20,0.0000,OK
하이브리드 VLM,2636,1803,0.316,6,0.059,1.92,0.0823,OK (실측 8/8p)


[8] 8. QA가이드 (p.38).pdf   (38p)


,전체문자수,순수문자수,마크업비율,마크다운표수,행중복률,실행시간(s),비용(USD),상태
로더,,,,,,,,
PyPDF,24002,18830,0.215,0,NaN,2.27,0.0000,OK
PDFPlumber,23923,18839,0.213,0,NaN,5.14,0.0000,OK
PyMuPDF,22276,18830,0.155,0,NaN,0.11,0.0000,OK
PyMuPDF+Markdown,46780,35539,0.240,30,0.158,5.58,0.0000,OK
하이브리드 VLM,62115,47095,0.242,34,0.177,11.60,0.9747,OK (실측 38/38p)


---
# 문서별 PDF 로더 결정. 결과 해석

### ①  로컬 로더로는 아예 못 읽는 문서

| 문서 | PyPDF | PDFPlumber | PyMuPDF | PyMuPDF+MD | **VLM** |
|---|---:|---:|---:|---:|---:|
| 3. 쇼라 소개서 | 0자 | 0자 | 0자 | 0자 | **10,361자 / 표 11개** |
| 5. 협력사시스템 사용법 | 6자 | 6자 | 6자 | 6자 | **8,738자 / 표 10개** |

전 페이지가 이미지 1장으로만 구성된 스캔형 PDF로 텍스트 레이어를 읽는 로더는 무엇을 써도 결과가 같음. 

**최종선택: VLM**

### ②  로컬로 읽히지만 표가 깨지는 문서 — 품질 vs 비용의 판단

| 문서 | 최고 로컬 (순수문자 / 표) | VLM (순수문자 / 표) | VLM 비용 |
|---|---|---|---:|
| 1. 협력사 운영 통합 안내서 | PyMuPDF+MD 10,826 / 21 | **16,060 / 23** | $0.26 |
| 2. Hmall 소개서 | PyMuPDF+MD 6,769 / 11 | **14,520 / 29** | $0.19 |
| 4. 광고 상품 소개서 | PyMuPDF+MD 17,322 / 24 | **18,605 / 23** | $0.40 |
| 6. 신규 협력사 입점 절차 | PyMuPDF+MD 5,960 / 2 | **11,437 / 21** | $0.10 |
| 7. 데이터 영역 광고 제안서 | PyMuPDF+MD 1,299 / 4 | **1,802 / 6** | $0.05 |

4번 광고 상품 소개서는  표만 보면 PyMuPDF+Markdown이 전체문자수 39,886자로 VLM(26,746자)보다 많고 표도 24개 대 23개로 앞선 것처럼 보인다.
하지만 실제 출력을 열어보면 아래와 같다.

```
|~~**현**~~**대**<br>~~**백화점**~~|~~**탭**~~<br>**메인**<br>**A**<br>**B**<br>C<br>투~~뎁스~~<br>D<br>E|**_상단기**~~**획**~~**전**|50|만원<br>1일|만원<br>1일|
```

셀이 글자 단위로 쪼개지고, 구분 열 전체가 8개 행에 통째로 복사됐고, 구좌수 열은 사라졌다.
마크업비율이 **0.57**(전체의 57%가 마크업 기호)로 다른 문서의 0.24 수준보다 두 배 이상 높다.
**문자수가 많은 게 아니라 마크업이 많은 것**이다. 같은 페이지를 VLM으로 읽으면 이렇게 나온다.

```
| 구분 | 하위구분 | 상품명 | 단가(만원) | 노출기간 | 구좌수(日) |
|---|---|---|---|---|---|
| 현대백화점탭 | 메인 | A_상단기획전 | 50만원 | 1일 | 2개 |
| 현대백화점탭 | 투뎁스 | D_팝업 | 20/50만원 | 1일 | 무제한 |
```

6번 문서도 비슷하다. 로컬은 표를 2개밖에 못 만들었지만 VLM은 21개를 복원했다.

### ③  로컬로 충분한 문서 — VLM은 비용 낭비

| 문서 | 최고 로컬 (순수문자 / 표) | VLM (순수문자 / 표) | VLM 비용 |
|---|---|---|---:|
| 8. QA가이드 | PyMuPDF+MD **35,539 / 30** | 48,264 / 30 | **$0.99** |

표 개수가 30개로 동일하고, 마크업비율도 0.24로 정상이며, 행중복률(0.158)도 낮다.
텍스트 레이어가 22,239자로 온전히 살아 있는 일반 문서다.
VLM을 쓰면 회수량이 1.36배 늘지만 **$0.99 — 8개 문서 전체 비용의 절반**이 한 문서에 든다.
QA 문답 위주라 표가 단순해서 로컬 로더로 충분하다.

### 결론: 문서별 라우팅

로컬 표가 멀쩡한 문서까지 VLM으로 돌리면 비용만 소모된다.
실제 출력을 열어 표 품질을 확인한 뒤 아래처럼 결정.

```
1번 협력사 운영 안내서 → PyMuPDF+Markdown  (표 정상, $0.34 절약)
2~7번                → 하이브리드 VLM      (표 복원 필요 또는 로컬 불가)
8번 QA가이드          → PyMuPDF+Markdown  (일부 행 손실 감수, $0.99 절약)
```

| 구성 | 비용 |
|---|---:|
| 전부 VLM | $2.80 |
| **채택안 (2~7만 VLM)** | **$1.58** |

**4번을 로컬로 내리지 않은 이유.** 표 개수만 보면 로컬이 24개로 VLM(22개)보다 많아
로컬을 골라도 될 것처럼 보인다. 그런데 그 표의 실제 내용은 이렇다.

```
|구분|상품명|Col3|단가(만원)|노출<br>기간|Col6|
|~~**현**~~**대**<br>~~**백화점**~~|~~**탭**~~<br>**메인**<br>**A**<br>**B**<br>C<br>투~~뎁스~~<br>D<br>E|...|50|만원<br>1일|
```

글자가 쪼개지고, 2열에 `탭·메인·A·B·C·투뎁스·D·E`가 통째로 들어가 행 구분이 사라졌고,
`구좌수` 열은 `Col6`만 남고 값이 없다. **표 개수는 세어지지만 행×열 조회가 불가능하다.**
4번은 단가표 문서라 2단계 골든셋의 표 문항이 가장 많이 나오는 곳이어서 VLM을 유지했다.

**8번의 트레이드오프.** 로컬 표가 대체로 정상이지만 일부 행이 깨진다.

```
|**탕류**|아이스팩中1개|아이스팩中1개|...   ← 정상
|**튀김**|**튀김**|**튀김**|...             ← 12개월 값이 전부 품목명으로 덮임
```

튀김·포장육·수산물 행의 월별 기준값이 소실된다. 그래도 $0.99(전체의 35%)를 아낄 수 있어
손실을 알고 로컬을 선택했다.

---
# 1단계-B. 로더 라우팅 · 청킹 · 벡터스토어

`PDF_CONFIGS`에 적힌 대로 문서마다 다른 로더를 태운다.
어떤 로더로 읽었는지는 청크 메타데이터(`loader`)에 남겨서 나중에 추적할 수 있게 했다.

**청킹** — `MarkdownHeaderTextSplitter`(`#`, `##`)로 1차 분할해 문서 계층을
메타데이터로 보존하고, `RecursiveCharacterTextSplitter`(900자 / overlap 150)로 2차 분할한다.
`####`는 분할 기준에서 뺐다. VLM이 만드는 `#### 표 행 풀어쓰기`가
표 본문과 떨어지면 안 되기 때문이다.


In [4]:
# =====================================================================
# 1-B단계: 문서별 로더 라우팅 + 청킹 + 벡터스토어 구축
# ---------------------------------------------------------------------
# 위 벤치마크 결과에 따라 문서마다 로더를 다르게 쓴다.
# 로컬 로더로 충분한 문서에 VLM을 쓰면 비용만 나가고,
# 텍스트 레이어가 없는 문서는 로컬 로더로 0자가 나오기 때문이다.
#
#   "pymupdf_markdown" : 로컬, 비용 0
#   "vlm"              : 하이브리드 VLM (페이지 이미지 + 텍스트 레이어), 비용 발생
# =====================================================================
import re
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import pymupdf
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_core.documents import Document
from langchain_text_splitters import (
    MarkdownHeaderTextSplitter,
    RecursiveCharacterTextSplitter,
)
from langchain_chroma import Chroma

# ---------------------------------------------------------------------
# 문서별 로더 선택  ← 벤치마크 표를 보고 수정하는 지점
# ---------------------------------------------------------------------
PDF_CONFIGS = {
    # 문서번호: 로더        # 근거 (1-B 벤치마크 결과)
    1: "pymupdf_markdown",  # 협력사 운영 안내서 — 로컬 표 정상(마크업 0.244), $0.34 절약
    2: "vlm",               # Hmall 소개서 — 로컬 표 열 붕괴(행중복률 0.578)
    3: "vlm",               # 쇼라 소개서 — 로컬 0자(스캔형), 대안 없음
    4: "vlm",               # 광고 상품 소개서 — 로컬 단가표 셀 조각남(마크업 0.566)
    5: "vlm",               # 협력사시스템 사용법 — 로컬 6자(스캔형), 대안 없음
    6: "vlm",               # 신규 입점 절차 — 표 2개 vs 17개
    7: "vlm",               # 데이터 영역 광고 제안서 — 회수량 1.37배
    8: "pymupdf_markdown",  # QA가이드 — 일부 행 손실 감수, $0.99 절약
}


# 파일명 -> 문서번호 변환. PDF_CONFIGS 의 키를 찾는 데 쓴다.
def doc_number(pdf_path) -> int:
    """파일명 앞의 번호를 문서 식별자로 쓴다.

    파일명 전체를 키로 쓰면 `(p.21)` -> `(p.38)` 처럼 이름이 바뀌었을 때
    설정을 못 찾아 그 문서가 조용히 누락된다. 번호로 맞추면 그 일이 없다.
    """
    m = re.match(r"\s*(\d+)", Path(pdf_path).name)
    if not m:
        raise ValueError(f"파일명에서 문서번호를 찾을 수 없습니다: {pdf_path}")
    return int(m.group(1))


# 설정이 빠진 문서가 있으면 조용히 건너뛰지 말고 즉시 멈춘다
# 설정이 빠진 문서를 미리 잡아 즉시 멈춘다.
# 예전에는 설정이 없으면 조용히 건너뛰었는데, 그 탓에 8번 문서가
# 통째로 누락된 채 진행된 적이 있다. 에러 없이 사라지는 쪽이 더 위험하다.
_missing = [p.name for p in PDF_LIST if doc_number(p) not in PDF_CONFIGS]
if _missing:
    raise ValueError(f"PDF_CONFIGS에 설정이 없는 문서: {_missing}")


def load_pymupdf_markdown(pdf_path):
    """로컬 로더. 텍스트 레이어가 살아 있고 표가 단순한 문서용."""
    # mode="page"           : 페이지 단위로 Document 를 만든다(문서 전체를 한 덩어리로 받지 않음)
    # extract_tables="markdown": 표를 마크다운 파이프 테이블로 변환해 준다
    docs = PyMuPDFLoader(str(pdf_path), mode="page",
                         extract_tables="markdown").load()
    return [
        # 메타데이터를 VLM 경로와 똑같은 형식으로 맞춘다.
        # 두 로더의 결과가 뒤섞여 한 벡터스토어에 들어가므로
        # 키 이름이 다르면 나중에 출처 추적이 깨진다.
        # PyMuPDF 의 page 는 0부터 시작하므로 +1 해서 사람이 보는 쪽수와 맞춘다.
        Document(page_content=d.page_content,
                 metadata={"source": Path(pdf_path).name,
                           "page": d.metadata.get("page", i) + 1,
                           "loader": "pymupdf_markdown"})
        for i, d in enumerate(docs) if d.page_content.strip()
    ]


def load_vlm(pdf_path):
    """하이브리드 VLM. 1-A 셀의 vlm_parse_page를 그대로 재사용한다(캐시 공유)."""
    with pymupdf.open(pdf_path) as d:
        n = d.page_count
    # 페이지마다 API 를 부르므로 8개씩 동시에 처리한다.
    # 대부분 네트워크 대기 시간이라 스레드로도 충분히 빨라진다.
    # ex.map 은 입력 순서대로 결과를 돌려주므로 페이지 순서가 보장된다.
    with ThreadPoolExecutor(max_workers=8) as ex:
        out = list(ex.map(lambda i: vlm_parse_page(pdf_path, i), range(n)))
    return [
        Document(page_content=md,
                 metadata={"source": Path(pdf_path).name,
                           "page": i + 1,
                           "loader": "vlm"})
        for i, (md, *_rest) in enumerate(out) if md.strip()
    ]


LOADER_FN = {"pymupdf_markdown": load_pymupdf_markdown, "vlm": load_vlm}

# ---------------------------------------------------------------------
# 1. 문서별 로딩
# ---------------------------------------------------------------------
# 문서별로 지정된 로더를 태워 페이지 단위 Document 를 모은다.
raw_docs = []
for pdf_path in PDF_LIST:
    method = PDF_CONFIGS[doc_number(pdf_path)]   # 이 문서에 쓸 로더 이름
    docs = LOADER_FN[method](pdf_path)           # 이름 -> 실제 함수로 바꿔 호출
    chars = sum(len(d.page_content) for d in docs)
    print(f"  {method:18s} {pdf_path.name[:32]:34s} {len(docs):3d}p {chars:7,d}자")
    raw_docs += docs

print(f"\n📄 총 {len(raw_docs)}페이지 / "
      f"{sum(len(d.page_content) for d in raw_docs):,}자")

  pymupdf_markdown   1. 협력사 운영 통합 안내서 (p.23).pdf         23p  14,294자
  vlm                2. Hmall 소개서 (p.30).pdf             30p  19,651자
  vlm                3. 쇼라 소개서 (p.22).pdf                22p  14,945자
  vlm                4. 광고 상품 소개서 (p.24).pdf             24p  28,131자
  vlm                5. 협력사시스템 사용법 (p.10).pdf            10p  13,948자
  vlm                6. 신규 협력사 입점 절차 안내 (p.21).pdf       21p  17,373자
  vlm                7. 데이터 영역 광고 제안서 (p.8).pdf           8p   2,622자
  pymupdf_markdown   8. QA가이드 (p.38).pdf                 37p  46,743자

📄 총 175페이지 / 157,707자


In [5]:
# =====================================================================
# 파싱 결과 확인 — 문서별로 본문 일부 출력
# ---------------------------------------------------------------------
# 페이지 앞부분이 아니라 표가 나오는 부분을 보여준다.
# =====================================================================
SAMPLE_CHARS = 800   # 문서당 출력할 글자 수


# 페이지 앞에서 그냥 n글자를 자르면 제목과 설명문만 나오고
# 정작 확인하고 싶은 표가 화면 밖으로 밀린다. 그래서 표 위치를 찾아 거기서 자른다.
def pick_excerpt(text: str, n: int) -> str:
    """표가 있으면 표 앞 한두 줄부터, 없으면 문서 앞부터 n글자."""
    lines = text.splitlines()
    # 마크다운 표의 구분선(|---|---|)이 있는 줄 번호. 표가 없으면 None.
    sep = next((i for i, l in enumerate(lines) if "|---" in l), None)
    # 구분선 두 줄 위부터 시작한다(헤더 행과 표 제목까지 같이 보이도록).
    start = 0 if sep is None else max(0, sep - 2)   # 표 헤더 한 줄 위부터
    return "\n".join(lines[start:])[:n]


for num in sorted(PDF_CONFIGS):
    docs = [d for d in raw_docs if doc_number(d.metadata["source"]) == num]
    if not docs:
        print(f"[{num}] 로드된 페이지 없음\n")
        continue

    # 같은 문서의 페이지들은 로더·출처가 모두 같으므로 첫 페이지에서 뽑아 쓴다
    src = docs[0].metadata["source"]
    loader = docs[0].metadata["loader"]
    total = sum(len(d.page_content) for d in docs)

    # 로더 선택이 갈리는 지점은 '표'다. 표가 있는 페이지를 우선 후보로 삼고,
    # 그중 가장 긴 페이지를 고른다(내용이 많아야 품질 판단이 된다).
    # 표가 하나도 없으면 with_table 이 빈 리스트라 or 로 전체 페이지에서 고른다.
    with_table = [d for d in docs if "|---" in d.page_content]
    pick = max(with_table or docs, key=lambda d: len(d.page_content))
    body = pick.page_content.strip()
    excerpt = pick_excerpt(body, SAMPLE_CHARS)

    print("=" * 100)
    print(f"[{num}] {src}")
    print(f"     로더 {loader}  |  {len(docs)}페이지  |  총 {total:,}자  |  표 포함 페이지 {len(with_table)}/{len(docs)}")
    print(f"     ▼ p.{pick.metadata['page']} ({len(body):,}자) 중 {'표 부분' if with_table else '앞부분'} {len(excerpt)}자")
    print("-" * 100)
    print(excerpt + ("\n... (이하 생략)" if len(excerpt) < len(body) else ""))
    print()

[1] 1. 협력사 운영 통합 안내서 (p.23).pdf
     로더 pymupdf_markdown  |  23페이지  |  총 14,294자  |  표 포함 페이지 14/23
     ▼ p.16 (1,645자) 중 표 부분 800자
----------------------------------------------------------------------------------------------------

|구분|출고<br>기준일|방송<br>시간|방송<br>당일|방송<br>+1일|방송<br>+2일|방송<br>+3일|
|---|---|---|---|---|---|---|
|**일반상품/**<br>**일반식품**|**1일**|**주간**|**60%**|**40%**|**-**|**-**|
|**일반상품/**<br>**일반식품**|**1일**|**야간**|**-**|**100%**|**-**|**-**|
|**일반상품/**<br>**일반식품**|**2일**|**주간**|**60%**|**30%**|**10%**|**-**|
|**일반상품/**<br>**일반식품**|**2일**|**야간**|**-**|**90%**|**10%**|**-**|
|**일반상품/**<br>**일반식품**|**3일**|**주간**|**60%**|**20%**|**10%**|**10%**|
|**일반상품/**<br>**일반식품**|**3일**|**야간**|**-**|**80%**|**10%**|**10%**|
|**냉동/냉장**<br>**식품**|**3일**|**주간**|**50%**|**20%**|**20%**|**10%**|
|**냉동/냉장**<br>**식품**|**3일**|**야간**|**-**|**50%**|**30%**|**20%**|


|주문일|출고기준일|출고마감일|내용|
|---|---|---|---|
|**05.27(월)**|**1일**|**05.28(화)**|**05.27~28출고가능**|
|**05.27(월)**|**2일**|**05.29(수)**|**05.27~

In [6]:
import re

# ---------------------------------------------------------------------
# 1. 목차·표지 페이지 제외
#    목차는 '항목명 -> 페이지번호' 나열일 뿐 정보가 없음
#    벡터스토어에 들어가면 용량만 차지
#
#    청크로 쪼갠 뒤가 아니라 페이지 단위로 거른다.
#    쪼갠 뒤 거르면 목차 한 페이지가 여러 청크로 흩어져 일부만 남는다.
# ---------------------------------------------------------------------
# 목차·표지를 알아보는 네 가지 신호
MIN_PAGE_CHARS = 100                                  # 공백 제외 글자 수
# (1) 제목에 목차 키워드가 있는가. re.I 는 대소문자 무시(contents/CONTENTS 모두 매치)
TOC_HEAD   = re.compile(r"(목차|차례|CONTENTS|INDEX)", re.I)
# (2) 점선 리더가 있는가. 목차 특유의 "항목 ..... 쪽번호" 서식
DOT_LEADER = re.compile(r"\.{5,}\s*\d+\s*P", re.I)    # "회사개요 ......... 2P"
# (3) 쪽 번호 참조가 여러 번 나오는가. 장(章) 목차 표가 이 형태다
PAGE_REF   = re.compile(r"\b\d+\s*P\b", re.I)         # 표 안의 "3P", "15P"

# 문서별 마지막 페이지 번호 (맺음말 판정에 쓴다)
# (4) 위치 조건에 쓸 문서별 마지막 페이지 번호를 미리 구한다.
#     '감사합니다' 같은 맺음말은 항상 마지막 페이지에 있다.
_last_page = {}
for d in raw_docs:
    src = d.metadata["source"]
    _last_page[src] = max(_last_page.get(src, 0), d.metadata["page"])


def front_matter_reason(doc):
    """목차·표지면 제외 사유, 본문이면 None."""
    body = doc.page_content.strip()
    head = "\n".join(body.splitlines()[:6])   # 제목 줄에서만 목차 키워드를 찾는다
    if TOC_HEAD.search(head):                  # 본문에 '목차'가 언급된 페이지는 안 걸리게
        return "목차(제목)"
    # 점선 리더가 2줄 이상이면 목차다. 1줄은 우연히 나올 수 있어 기준을 2로 둔다.
    if len(DOT_LEADER.findall(body)) >= 2:
        return "목차(점선 리더)"
    # "3P", "15P" 같은 쪽 참조가 4개 이상이면 장 목차 표로 본다.
    # 본문에도 쪽 번호가 한두 개 나올 수 있어 기준을 넉넉히 잡았다.
    if len(PAGE_REF.findall(body)) >= 4:
        return "목차(페이지 참조 다수)"

    # 짧은 페이지는 '문서 앞 2장 또는 마지막 장'일 때만 표지·맺음말로 본다.
    # 위치 조건이 없으면 이미지 위주의 짧은 본문(예: Hmall 소개서 p.21
    # 플랫폼행사 특별지원구좌)까지 잘려나간다.
    page, src = doc.metadata["page"], doc.metadata["source"]
    if (len(re.sub(r"\s+", "", body)) < MIN_PAGE_CHARS
            and (page <= 2 or page == _last_page[src])):
        return "표지/맺음말"
    return None


# 전체 페이지를 본문(body_docs)과 제외 대상(dropped)으로 가른다.
# dropped 는 버리지 않고 들고 있다가 아래에서 사유와 함께 출력한다.
# 무엇이 왜 걸러졌는지 눈으로 봐야 과도한 필터링을 잡아낼 수 있기 때문이다.
body_docs, dropped = [], []
for d in raw_docs:
    reason = front_matter_reason(d)
    (dropped.append((d, reason)) if reason else body_docs.append(d))

print(f"🧹 목차·표지 제외: {len(raw_docs)} → {len(body_docs)}페이지 ({len(dropped)}개 제외)")
for d, reason in dropped:
    head = " ".join(d.page_content.split())[:46]
    print(f"     - {d.metadata['source'][:24]:26s} p.{d.metadata['page']:<3d} {reason:16s} | {head}")

🧹 목차·표지 제외: 175 → 159페이지 (16개 제외)
     - 1. 협력사 운영 통합 안내서 (p.23).   p.1   표지/맺음말           | 현대홈쇼핑 협력사운영통합안내서 2024.06
     - 1. 협력사 운영 통합 안내서 (p.23).   p.2   목차(제목)           | 목차 1. 현대홈쇼핑사업영역 2. 상품운영프로세스 3. 파트너시스템 4. 업무문의(
     - 1. 협력사 운영 통합 안내서 (p.23).   p.23  표지/맺음말           | 감사합니다. 2024.06
     - 2. Hmall 소개서 (p.30).pdf    p.1   목차(제목)           | # Hmall 소개서 ## CONTENTS Ⅰ. 회사개요 ..............
     - 2. Hmall 소개서 (p.30).pdf    p.2   목차(페이지 참조 다수)    | # Ⅰ. 회사개요 | 번호 | 항목 | 페이지 | |---|---|---| | 01
     - 2. Hmall 소개서 (p.30).pdf    p.9   목차(페이지 참조 다수)    | # Ⅱ. 입점및지원제도 | 번호 | 항목 | 페이지 | |---|---|---| |
     - 2. Hmall 소개서 (p.30).pdf    p.15  목차(페이지 참조 다수)    | # Ⅲ. PC · 모바일· 라이브플랫폼 | 번호 | 내용 | 페이지 | |---|-
     - 3. 쇼라 소개서 (p.22).pdf       p.2   목차(제목)           | # 목차구성 CONTENTS COMPOSITION 믿고사 쇼라 ## 목차 | 번호 
     - 4. 광고 상품 소개서 (p.24).pdf    p.1   표지/맺음말           | # 광고상품소개서 현대홈쇼핑 Hmall Hmall 광고 상품 소개서 2026.08
     - 4. 광고 상품 소개서 (p.24).pdf    p.24  표지/맺음말           | # 감사합니다


In [7]:

# ---------------------------------------------------------------------
# 2. 청킹
#    마크다운 헤더로 1차 분할해 계층 구조를 메타데이터로 보존하고,
#    문자 수 기준으로 2차 분할한다.
#    '####'는 분할 기준에서 뺐다 — `#### 표 행 풀어쓰기`가 표 본문과 떨어지면 안 된다.
# ---------------------------------------------------------------------
# 1차 분할기: 마크다운 제목(#, ##)을 만나면 끊는다.
#   strip_headers=False -> 제목 줄을 청크 안에 남긴다.
#   제목이 사라지면 "무엇에 대한 표인지" 모르는 청크가 되기 때문이다.
#   분할 결과의 metadata에 h1/h2 값이 들어가 나중에 출처 추적에 쓰인다.
header_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[("#", "h1"), ("##", "h2")], strip_headers=False)

# 2차 분할기: 1차 결과가 900자를 넘으면 글자 수 기준으로 더 쪼갠다.
#   chunk_overlap=150 -> 앞뒤 청크가 150자씩 겹친다.
#   경계에서 잘린 문장이 어느 한쪽에는 온전히 남도록 하는 장치다.
#   separators 순서가 중요하다. 문단(\n\n) -> 줄(\n) -> 공백 -> 글자 순으로
#   시도하므로, 가능하면 문단 경계에서 끊고 최후에만 글자 단위로 자른다.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=900, chunk_overlap=150, separators=["\n\n", "\n", " ", ""])

# 1차 분할 실행. body_docs 는 목차·표지를 걸러낸 페이지 목록이다.
header_docs = []
for d in body_docs:
    for s in header_splitter.split_text(d.page_content):
        # split_text 가 만든 조각은 h1/h2 만 갖고 있어 출처를 모른다.
        # 원본 페이지의 metadata(source/page/loader)를 먼저 깔고 그 위에 덮어써서
        # "어느 문서 몇 페이지의 어느 절" 정보를 모두 갖게 한다.
        s.metadata = {**d.metadata, **s.metadata}
        header_docs.append(s)

# 2차 분할 후 30자 미만 조각은 버린다.
# 표 끝부분이나 제목만 남은 파편은 검색에 걸려도 답이 되지 못하고
# 오히려 상위 k개 자리를 차지해 진짜 근거를 밀어낸다.
all_chunks = [c for c in text_splitter.split_documents(header_docs)
              if len(c.page_content.strip()) >= 30]
print(f"✨ 최종 청크 {len(all_chunks)}개 "
      f"(평균 {sum(len(c.page_content) for c in all_chunks)//max(len(all_chunks),1)}자)")

✨ 최종 청크 349개 (평균 463자)


---
# 1단계-C. 벡터스토어 선택

### 벡터스토어 선택: Chroma

같은 임베딩을 사용하므로 네개 스토어의 검색 결과는 사실상 같음
**즉 "어느 게 더 잘 찾나"로는 결정할 수 없음.** 

| | InMemory | **Chroma** | FAISS | Qdrant |
|---|---|---|---|---|
| 디스크 영속성 | ✗ 종료 시 소멸 | **○ SQLite 자동** | △ 수동 save/load | △ 서버 or 파일 설정 |
| 메타데이터 필터 | △ 기본 | **○ where 절 직관적** | △ 제한적 | ○ 강력 |
| 설치·운영 | 없음 | **pip 한 줄** | pip | pip(로컬) / 서버(운영) |
| 적합 규모 | 실험 | **로컬~중소** | 대용량 단일 노드 | 대규모·분산 |

**Chroma를 택한 이유**

1. **영속성이 기본이다.** `persist_directory`만 주면 SQLite로 저장되고 다음 실행에서
   그대로 불러온다. 이 프로젝트는 파싱에 실제 비용($1.58)이 들었으므로
   커널을 껐다고 다시 임베딩하는 일이 없어야 한다. InMemory는 이 점에서 탈락.
2. **메타데이터 필터가 직관적이다.** 청크에 `source`·`page`·`loader`를 넣어뒀고,
   "특정 문서 안에서만 검색" 같은 조건을 `where`로 바로 걸 수 있다.
3. FAISS는 검색이 빠르지만 저장/복원을 직접 관리해야 하고, Qdrant는 기능이
   가장 풍부하지만 이 규모에서는 과하다.


In [ ]:
# =====================================================================
# 3-A. 벡터스토어 4종 비교
# ---------------------------------------------------------------------
# InMemory / Chroma / FAISS / Qdrant 에 같은 청크를 넣고 비교한다.
#
# 주의할 점이 두 가지 있다.
#  1) 네개의 스토어 모두 같은 임베딩(text-embedding-3-small)을 쓰므로
#     검색 품질 자체는 거의 같고, 속도·영속성·운영 편의에서 달라짐
#     "어느 게 더 잘 찾나"가 아니라 "어느 게 우리 상황에 맞나"를 보고 판단.
#  2) 스토어마다 점수 의미가 다르다.
#     InMemory·Qdrant는 유사도(클수록 가까움), FAISS·Chroma는 거리(작을수록 가까움).
#     그래서 점수 숫자를 직접 비교하지 않고, 검색된 '문서'가 같은지를 본다.
# =====================================================================
import time
import pandas as pd

from langchain_core.vectorstores import InMemoryVectorStore
from langchain_community.vectorstores import FAISS
from langchain_qdrant import QdrantVectorStore
from langchain_core.embeddings import Embeddings


class CachedEmbeddings(Embeddings):
    """같은 청크를 4개 스토어에 넣을 때 임베딩 API를 한 번만 호출한다.

    이게 없으면 인덱싱 시간의 대부분이 임베딩 API 대기라서
    스토어 자체의 색인 속도 차이가 묻힌다. 비용도 4배가 된다.

    FAISS가 Embeddings 서브클래스가 아니면 '호출 가능한 함수'로 오해하므로
    반드시 langchain_core.embeddings.Embeddings 를 상속해야 한다.
    """

    def __init__(self, base):
        self.base = base
        self._cache = {}

    def embed_documents(self, texts):
        # 아직 임베딩하지 않은 텍스트만 골라 API 를 부른다.
        # 두 번째 스토어부터는 new 가 비어 있어 API 호출이 아예 없다.
        new = [t for t in texts if t not in self._cache]
        if new:
            for t, v in zip(new, self.base.embed_documents(new)):
                self._cache[t] = v
        return [self._cache[t] for t in texts]

    def embed_query(self, text):
        return self.base.embed_query(text)


cached_emb = CachedEmbeddings(embeddings)
# 비교 시작 전에 미리 전량 임베딩해 캐시를 채운다.
# 그래야 아래에서 재는 '인덱싱 시간'이 순수하게 스토어의 색인 속도가 된다.
cached_emb.embed_documents([c.page_content for c in all_chunks])   # 1회만 임베딩
print(f"임베딩 준비 완료: 청크 {len(all_chunks)}개\n")

# 비교용 스토어는 전부 메모리에 만든다.
# 실제 서비스용 ./chroma_db 를 건드리지 않고, 재실행 시 중복 적재도 막는다.
BUILDERS = {
    "InMemory": lambda docs, emb: InMemoryVectorStore.from_documents(docs, emb),
    "Chroma":   lambda docs, emb: Chroma.from_documents(
                    docs, emb, collection_name="vs_bench_chroma"),
    "FAISS":    lambda docs, emb: FAISS.from_documents(docs, emb),
    "Qdrant":   lambda docs, emb: QdrantVectorStore.from_documents(
                    docs, emb, location=":memory:",
                    collection_name="vs_bench_qdrant"),
}

TEST_QUERIES = [
    "현대백화점탭 투뎁스 D_팝업 광고의 단가는 얼마인가?",
    "신규 협력사가 입점하려면 어떤 절차를 거쳐야 하나요?",
    "탕류의 7월 아이스팩 기준은 무엇인가?",
    "쇼라 방송 제작에 참여하는 전문 인력은?",
    "출고기준일과 출고마감일은 어떻게 산정하나요?",
]
K = 4

rows, hits_by_store = [], {}
for name, build in BUILDERS.items():
    # 인덱싱 시간: 청크를 스토어에 넣고 색인을 만드는 데 걸린 시간
    t0 = time.perf_counter()
    store = build(all_chunks, cached_emb)
    index_s = time.perf_counter() - t0

    # 검색 시간: 질문 5개 평균. 질문 임베딩(API 왕복)이 포함되므로
    # 스토어별 차이보다 네트워크 지연이 더 크게 나온다.
    t0 = time.perf_counter()
    hits = [store.similarity_search(q, k=K) for q in TEST_QUERIES]
    search_ms = (time.perf_counter() - t0) / len(TEST_QUERIES) * 1000

    hits_by_store[name] = [
        [(d.metadata.get("source"), d.metadata.get("page")) for d in h] for h in hits
    ]
    rows.append({"스토어": name,
                 "인덱싱(s)": round(index_s, 2),
                 "검색 평균(ms)": round(search_ms, 1)})

# ---------- 검색 결과가 서로 같은가 (Chroma 기준) ----------
# 점수는 스토어마다 의미가 달라(유사도 vs 거리) 직접 비교할 수 없다.
# 대신 '같은 문서를 찾아왔는가'를 센다.
# 실제로 채택한 Chroma 를 기준으로 삼아 나머지와 교집합 크기를 구한다.
base = hits_by_store["Chroma"]
for r in rows:
    mine = hits_by_store[r["스토어"]]
    # 질문별 top-K 집합의 교집합 크기를 모두 더한다 (최대 K × 질문 수)
    overlap = sum(len(set(a) & set(b)) for a, b in zip(mine, base))
    r["Chroma와 top-4 일치"] = f"{overlap}/{K * len(TEST_QUERIES)}"

df_vs = pd.DataFrame(rows)
print("===== 벡터스토어 4종 비교 =====")
display(df_vs)

# ---------- 같은 질문에 무엇을 찾았는지 ----------
print(f"\n질문: {TEST_QUERIES[0]}")
for name in BUILDERS:
    top = hits_by_store[name][0]
    print(f"  {name:9s} " + " | ".join(f"{s[:14]} p.{p}" for s, p in top[:3]))

print("""
※ 네 스토어의 검색 결과가 거의 같은 것은 정상이다. 임베딩이 같으면
   최근접 이웃도 같기 때문이다. 따라서 선택 기준은 검색 품질이 아니라
   영속성·메타데이터 필터·운영 부담이다.""")

임베딩 준비 완료: 청크 349개

===== 벡터스토어 4종 비교 =====


,스토어,인덱싱(s),검색 평균(ms),Chroma와 top-4 일치
0,InMemory,0.01,466.4,17/20
1,Chroma,1.53,248.4,17/20
2,FAISS,0.45,206.0,17/20
3,Qdrant,1.33,259.5,17/20



질문: 현대백화점탭 투뎁스 D_팝업 광고의 단가는 얼마인가?
  InMemory  4. 광고 상품 소개서 ( p.14 | 4. 광고 상품 소개서 ( p.4 | 4. 광고 상품 소개서 ( p.15
  Chroma    4. 광고 상품 소개서 ( p.14 | 4. 광고 상품 소개서 ( p.4 | 4. 광고 상품 소개서 ( p.15
  FAISS     4. 광고 상품 소개서 ( p.14 | 4. 광고 상품 소개서 ( p.4 | 4. 광고 상품 소개서 ( p.15
  Qdrant    4. 광고 상품 소개서 ( p.14 | 4. 광고 상품 소개서 ( p.4 | 4. 광고 상품 소개서 ( p.15

※ 네 스토어의 검색 결과가 거의 같은 것은 정상이다. 임베딩이 같으면
   최근접 이웃도 같기 때문이다. 따라서 선택 기준은 검색 품질이 아니라
   영속성·메타데이터 필터·운영 부담이다.


In [9]:

# ---------------------------------------------------------------------
# 3. Chroma 재구축 (이전 파싱 결과가 섞이지 않게 컬렉션을 비우고 적재)
# ---------------------------------------------------------------------
# 기존 컬렉션을 통째로 비운다.
# from_documents 는 기존 데이터를 지우지 않고 '덧붙이므로', 이 줄이 없으면
# 셀을 다시 실행할 때마다 같은 청크가 중복 적재되어 검색 결과에
# 동일 문서가 여러 번 뜬다.
Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings,
       collection_name=COLLECTION_NAME).reset_collection()

# 청크를 임베딩해 Chroma에 적재한다.
#   persist_directory 를 주면 SQLite 파일로 디스크에 저장되어
#   커널을 껐다 켜도 다시 임베딩할 필요가 없다(임베딩 비용 절약).
vectorstore = Chroma.from_documents(
    documents=all_chunks, embedding=embeddings,
    persist_directory=CHROMA_PATH, collection_name=COLLECTION_NAME)

print(f"🚀 Chroma 구축 완료 → {CHROMA_PATH} ({vectorstore._collection.count()}개 벡터)")

🚀 Chroma 구축 완료 → ./chroma_db (349개 벡터)


---
# 2단계. 골든 데이터셋 생성 및 큐레이션

총 22문항. 8개 유형으로 나눠 검색·생성 능력을 각각 다른 각도에서 본다.

| 유형 | 문항 | 의도 |
|---|---:|---|
| 단일홉 | 4 | 근거 한 곳으로 답하는 기본 검색 |
| 멀티홉 | 3 | 서로 다른 페이지 두 곳을 결합해야 답이 나옴 |
| 표/절차 | 3 | 표에서 행 조건에 맞는 값을 골라내는가 |
| 집계 | 3 | 열거된 항목을 빠짐없이 세거나 나열하는가 |
| 조건 | 2 | 무엇을 하려면 무엇이 필요한가 / 분류별로 조건이 어떻게 갈리는가 |
| 부정/예외 | 2 | 제외·불가 조건 |
| 시점/기한 | 2 | 기한·처리기간·D-day |
| 무답변 | 3 | 문서에 없는 질문에 지어내지 않는가 |
| **합계** | **22** | |

### 설계 포인트

**① 표 문항의 근거를 키워드가 아니라 표 구조로 지목한다.**
파싱 결과에 마크다운 표가 들어 있으므로 `"|---" in chunk`로 표 청크를 정확히 고른다.
단, **3열 미만 표는 제외**한다 — 1열짜리 목록으로는 "행×열 교차" 질문을 만들 수 없고,
정답이 목록 일부만 담게 되어 골든셋 자체가 틀린다.

**② 표 문항은 행 조건을 2개 이상 요구한다.**
`구분`과 `하위구분`을 모두 지정해야 답이 하나로 특정되는 질문만 만든다.
표 전체를 나열시키는 질문은 행/열 매핑 능력을 측정하지 못한다.

**③ 목차·표지 청크를 근거에서 제외한다.**
목차는 '항목명 → 페이지번호' 나열일 뿐이라 "몇 페이지를 보면 되는가" 같은 문항이 나온다.

**④ 같은 표·같은 청크를 두 번 쓰지 않는다.**
같은 표가 여러 페이지에 반복되면 동일한 질문이 중복 생성된다.
표 헤더를 시그니처로 삼아 중복을 제거하고, 근거 청크도 유형 간 재사용을 막는다.

**⑤ 3단계 자동 검수**
- **인용 정합성** — 초안의 인용구가 근거에 한 글자도 다르지 않게 존재하는지 검사.
  멀티홉은 근거가 2개이므로 인용구도 근거별로 따로 받아 각각 대조한다
  (한 문자열로 합치게 하면 모델이 `/`로 이어붙여 어느 근거와도 일치하지 않는다).
- **정답 실질성** — 표의 빈 셀을 물어 정답이 "빈칸"이 되는 문항을 제거한다.
- **자기완결성** — 질문만 단독으로 읽어도 의도가 특정되는지 LLM이 판정한다.

**⑥ 무답변 문항을 하드코딩하지 않고 검증한다.**
후보 질문마다 검색을 돌려 상위 청크로 답이 가능한지 심사하고, **불가 판정된 것만** 채택한다.

In [15]:
# =====================================================================
# 2단계: 골든 데이터셋 생성 및 큐레이션
# ---------------------------------------------------------------------
# 목표 22문항
# 1단계 파싱이 표를 마크다운 테이블 + '표 행 풀어쓰기' 형태로 남기므로,
# 표 문항의 근거 청크를 키워드 추측이 아니라 실제 표 구조로 정확히 지목할 수 있다.
# =====================================================================
import re, json, random
from typing import Literal, List
from pydantic import BaseModel, Field

from langchain_chroma import Chroma
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.documents import Document

# MODEL / JUDGE / CHROMA_PATH / COLLECTION_NAME / GOLDEN_PATH / embeddings 는 공통 설정 셀에서 온다
random.seed(42)

# 유형별 목표 문항 수 (총 22)
TARGET = {
    "단일홉":   4,   # 근거 한 곳으로 답하는 기본 검색
    "멀티홉":   3,   # 서로 다른 페이지 두 곳을 결합해야 답이 나옴
    "표/절차":  3,   # 표에서 행 조건에 맞는 값을 골라내는가
    "집계":     3,   # 열거된 항목을 빠짐없이 세거나 나열하는가
    "조건":     2,   # 무엇을 하려면 무엇이 필요한가 / 분류별로 조건이 어떻게 갈리는가
    "부정/예외": 2,   # 제외·불가 조건
    "시점/기한": 2,   # 기한·처리기간·D-day
    "무답변":   3,   # 문서에 없는 질문에 지어내지 않는가
}
# 검수에서 절반 이상이 탈락하므로 목표보다 넉넉히 만들어 둔다.
# 예: 단일홉 목표 4개 x 3배 = 초안 12개 생성 -> 검수 후 4개 선택
OVERSAMPLE = 3
OVERSAMPLE_TABLE = 5   # 표 문항은 시그니처 중복 제거 후 후보가 줄어 더 넉넉히 뽑는다
print(f"🎯 목표 {sum(TARGET.values())}문항 / 유형별 {TARGET} (오버샘플 {OVERSAMPLE}배)")

# ---------- 1. 청크 로드 ----------
print("\n[1/5] Chroma DB 로드")
gold_store = Chroma(persist_directory=CHROMA_PATH, embedding_function=embeddings,
                    collection_name=COLLECTION_NAME)
data = gold_store.get()
# 목차·표지는 1단계 청킹에서 이미 제외했으므로 여기서는 길이만 본다
body = [Document(page_content=t, metadata=m or {})
        for t, m in zip(data["documents"], data["metadatas"])
        if t and len(t.strip()) > 150]
print(f"✅ 유효 본문 청크 {len(body)}개")

# ---------- 2. 유형별 근거 샘플링 ----------
print("\n[2/5] 유형별 근거 청크 샘플링")

MIN_TABLE_COLS = 3

def has_table(c):
    """행×열 교차 조회가 가능한 표를 가진 청크만 True.

    구분선(|---|---|)의 열 개수가 3개 미만이면 사실상 1열짜리 목록이라
    '어느 행의 어느 열' 형태의 질문을 만들 수 없다. 이런 표로 문항을 만들면
    정답이 목록 일부만 담게 되어 골든셋 자체가 틀린다(실제로 한 번 발생했다).
    """
    for line in c.page_content.splitlines():
        line = line.strip()
        if line.startswith("|") and set(line) <= set("|-: "):
            if line.count("|") - 1 >= MIN_TABLE_COLS:
                return True
    return False

# 키워드 중 하나라도 들어간 청크를 n개 고른다.
def having_any(keywords, n, pool=None):
    matched = [c for c in (pool or body) if any(k in c.page_content for k in keywords)]
    # 섞지 않으면 항상 문서 앞쪽 청크만 뽑혀 특정 문서에 문항이 쏠린다.
    # 위에서 random.seed(42) 를 고정했으므로 섞어도 매번 같은 결과가 나온다.
    random.shuffle(matched)
    return matched[:n]

table_pool = [c for c in body if has_table(c)]
plain_pool = [c for c in body if not has_table(c)]
print(f"   표 포함 청크 {len(table_pool)}개 / 일반 청크 {len(plain_pool)}개")

# 단일홉: 일반 청크에서 균등 간격
# 단일홉: 표가 없는 일반 청크에서 '균등 간격'으로 뽑는다.
# 앞에서부터 순서대로 자르면 1번 문서에서만 문항이 나오므로,
# 전체를 step 간격으로 건너뛰며 문서 전반에 퍼지게 한다.
step = max(1, len(plain_pool) // (TARGET["단일홉"] * OVERSAMPLE))
single_chunks = plain_pool[::step][:TARGET["단일홉"] * OVERSAMPLE]

# 멀티홉: 유사도 검색으로 연관 청크 짝짓기
pairs = []
for i in range(0, len(body), max(1, len(body) // (TARGET["멀티홉"] * OVERSAMPLE))):
    base = body[i]
    # 같은 페이지 조각끼리 묶으면 '멀티홉'이 되지 않으므로 다른 페이지에서 짝을 찾는다
    # 기준 청크와 의미가 가까운 청크를 8개 찾아 그중 첫 후보를 짝으로 삼는다.
    # 조건 두 가지: 자기 자신이 아니고, 페이지가 달라야 한다.
    # 같은 페이지 조각끼리 묶으면 한쪽만 봐도 답이 나와 멀티홉이 성립하지 않는다.
    partner = next((d for d in gold_store.similarity_search(base.page_content, k=8)
                    if d.page_content != base.page_content
                    and d.metadata.get("page") != base.metadata.get("page")), None)
    if partner:
        pairs.append((base, partner))
pairs = pairs[:TARGET["멀티홉"] * OVERSAMPLE]

# 표/절차: 같은 표가 여러 페이지에 반복 등장하므로 '표 시그니처'로 먼저 중복을 없앤다.
# (같은 표를 여러 번 뽑으면 똑같은 질문이 반복 생성된다)
def table_signature(c):
    """표의 헤더 행을 표 고유 식별자로 쓴다."""
    lines = c.page_content.splitlines()
    for i, line in enumerate(lines):
        st = line.strip()
        if st.startswith("|") and set(st) <= set("|-: ") and i > 0:
            return re.sub(r"\s+", "", lines[i - 1])   # 구분선 바로 위 = 헤더 행
    return re.sub(r"\s+", "", c.page_content[:80])

# 같은 표가 여러 페이지에 반복 등장하면 똑같은 질문이 중복 생성된다.
# 헤더 행을 지문 삼아 처음 나온 표만 남긴다.
uniq_tables, seen_sig = [], set()
for c in table_pool:
    sig = table_signature(c)
    if sig not in seen_sig:
        seen_sig.add(sig)
        uniq_tables.append(c)
print(f"   표 청크 중복 제거: {len(table_pool)} -> {len(uniq_tables)}개 고유 표")

# 여러 문서에 고루 퍼지도록 source 기준 라운드로빈
by_source = {}
for c in uniq_tables:
    by_source.setdefault(c.metadata.get("source", "?"), []).append(c)
for v in by_source.values():
    random.shuffle(v)

table_chunks, idx = [], 0
need = TARGET["표/절차"] * OVERSAMPLE_TABLE
while len(table_chunks) < need:
    added = False
    for v in by_source.values():
        if idx < len(v):
            table_chunks.append(v[idx]); added = True
            if len(table_chunks) >= need:
                break
    if not added:
        break
    idx += 1

# 집계: 항목이 여러 개 열거된 청크. 표 데이터 행·①②③·번호 목록을 모두 센다.
def enum_count(c):
    """청크 안에 열거된 항목 수. 개수를 묻는 질문을 만들 수 있는지 판단한다."""
    t = c.page_content
    rows = sum(1 for l in t.splitlines()
               if l.strip().startswith("|") and not set(l.strip()) <= set("|-: "))
    circled  = len(re.findall(r"[\u2460-\u246E]", t))       # ①~⑮
    numbered = len(re.findall(r"^\s*\d+[.)]\s", t, re.M))   # "1. ", "2) "
    return max(rows - 1, circled, numbered)                # 표는 헤더 1행 제외

agg_pool = [c for c in body if enum_count(c) >= 4]
random.shuffle(agg_pool)
count_chunks = agg_pool[:TARGET["집계"] * OVERSAMPLE]

# 조건: 무엇을 하려면 무엇이 필요한지, 분류에 따라 조건이 어떻게 달라지는지
condition_chunks = having_any(["조건", "기준", "경우", "해당하는", "필요", "이상", "미만"],
                              TARGET["조건"] * OVERSAMPLE)

# 부정/예외: 제외·불가 조건
negation_chunks = having_any(["제외", "불가", "않습니다", "없습니다", "단,", "유의"],
                             TARGET["부정/예외"] * OVERSAMPLE, plain_pool)

# 시점/기한: 기한·처리기간·D-day
temporal_chunks = having_any(["기한", "까지", "이내", "D-", "영업일", "익일", "마감"],
                             TARGET["시점/기한"] * OVERSAMPLE)

raw_jobs  = [("단일홉", [c]) for c in single_chunks]
raw_jobs += [("멀티홉", list(p)) for p in pairs]
raw_jobs += [("표/절차", [c]) for c in table_chunks]
raw_jobs += [("집계", [c]) for c in count_chunks]
raw_jobs += [("조건", [c]) for c in condition_chunks]
raw_jobs += [("부정/예외", [c]) for c in negation_chunks]
raw_jobs += [("시점/기한", [c]) for c in temporal_chunks]

# 같은 근거 청크가 두 번 쓰이면 사실상 같은 질문이 중복 생성된다
# 유형별 샘플링이 겹쳐 같은 청크가 두 유형에 모두 뽑히는 경우가 있다.
# 그대로 두면 사실상 같은 질문이 유형만 달리해 두 번 만들어진다.
# 청크 내용 해시를 키로 삼아 먼저 잡힌 유형 하나만 남긴다.
jobs, used = [], set()
for kind, cs in raw_jobs:
    key = tuple(sorted(hash(c.page_content) for c in cs))
    if key in used:
        continue
    used.add(key)
    jobs.append((kind, cs))
print(f"   생성할 초안 작업 {len(jobs)}개")

🎯 목표 22문항 / 유형별 {'단일홉': 4, '멀티홉': 3, '표/절차': 3, '집계': 3, '조건': 2, '부정/예외': 2, '시점/기한': 2, '무답변': 3} (오버샘플 3배)

[1/5] Chroma DB 로드
✅ 유효 본문 청크 279개

[2/5] 유형별 근거 청크 샘플링
   표 포함 청크 91개 / 일반 청크 188개
   표 청크 중복 제거: 91 -> 56개 고유 표
   생성할 초안 작업 61개


In [16]:
# ---------- 3. 초안 생성 ----------
print("\n[3/5] 초안 생성")

# 생성 결과를 받을 스키마.
# with_structured_output 에 넘기면 모델이 이 형식에 맞춰 JSON 을 반환하므로
# 응답 텍스트를 직접 파싱할 필요가 없고, 필드 누락도 막을 수 있다.
class Draft(BaseModel):
    question: str = Field(description="질문만 읽어도 무엇을 묻는지 특정되어야 함. '이 규정', '해당 문서' 같은 지시대명사 금지.")
    answer: str   = Field(description="근거에 기반한 정확한 정답")
    quotes: List[str] = Field(
        description="근거에서 **한 글자도 바꾸지 않고 그대로 복사한** 문장들. "
                    "근거가 2개면 각 근거에서 1개씩 총 2개. 여러 문장을 '/'나 따옴표로 "
                    "이어 붙이지 말고 리스트의 별도 항목으로 담아라. 요약·재작성 금지.")
    difficulty: Literal["easy", "medium", "hard"]

GUIDES = {
    "단일홉": "근거 한 곳만 보면 답할 수 있는 명확한 질문을 만든다.",
    "멀티홉": "두 근거를 모두 결합해야만 답할 수 있는 질문을 만든다. 한쪽만 봐서는 답이 나올 수 없어야 한다.",
    "표/절차": ("표에서 **특정 행과 특정 열이 교차하는 값**을 묻는 질문을 만든다. "
              "예: '어떤 구분의 어떤 상품의 단가는 얼마인가'처럼 행을 특정하는 조건을 2개 이상 넣어라. "
              "표 전체를 나열시키는 질문이나 열 이름만 묻는 질문은 만들지 마라. "
              "절차 표라면 특정 단계에서 무엇을 하는지 묻는다. "
              "값이 비어 있는 셀은 절대 묻지 마라 — 정답이 '빈칸'이 되는 질문은 금지."),
    "집계": ("근거에 열거된 항목이 몇 개인지 세거나 전부 나열하게 하는 질문을 만든다. "
           "답에는 개수와 항목 목록을 함께 쓴다. "
           "근거에 실제로 적힌 항목만 세고, 추측으로 더하거나 빼지 마라."),
    "조건": ("무엇을 하기 위해 충족해야 하는 조건이나 필요한 것을 묻는 질문을 만든다. "
           "또는 구분되는 분류 항목에 따라 조건이 어떻게 달라지는지를 묻는다. "
           "조건이 근거에 명시되어 있어야 하고, 답에 그 조건을 그대로 옮긴다."),
    "부정/예외": "무엇을 하면 안 되는지, 어떤 예외·제외 조건이 있는지를 묻는 질문을 만든다.",
    "시점/기한": "신청 기한, 처리 기간, D-day, 영업일 등 시간 조건을 묻는 질문을 만든다.",
}

# 질문 생성 체인. {guide} 자리에 유형별 지침이, {context} 자리에 근거 청크가 들어간다.
# temperature=0.2 : 0으로 두면 비슷한 근거에서 매번 같은 문장이 나와 문항이 단조로워지고,
#                   높이면 근거에 없는 내용을 지어낼 위험이 커진다. 그 사이 값.
writer_chain = ChatPromptTemplate.from_messages([
    ("system", "제공된 규정/매뉴얼 문서를 근거로 평가용 질문과 답을 작성합니다. "
               "근거에 없는 사실은 절대 작성하지 마세요.\n{guide}"),
    ("human", "근거 문서:\n{context}"),
]) | ChatOpenAI(model=MODEL, temperature=0.2).with_structured_output(Draft)

# 초안 생성. 한 건이 실패해도 전체가 멈추지 않도록 예외를 잡아 기록만 남긴다.
drafts, failed = [], []
for kind, cs in jobs:
    try:
        d = writer_chain.invoke({"guide": GUIDES[kind],
                                 "context": "\n---\n".join(c.page_content for c in cs)})
        drafts.append((kind, cs, d))
    except Exception as e:
        failed.append((kind, type(e).__name__))
        continue

print(f"   초안 {len(drafts)}개 생성" + (f" (실패 {len(failed)}개)" if failed else ""))

# ---------- 유형별 초안 확인 ----------
# 검수 전 원본. 유형별로 몇 개 확보했는지, 가이드대로 만들어졌는지 확인.
# 목표보다 적으면 뒤 검수에서 더 깎이므로 OVERSAMPLE을 올려야 함.
from collections import Counter
made = Counter(k for k, _, _ in drafts)

print("\n───────── 유형별 초안 현황 ─────────")
for kind in TARGET:
    if kind == "무답변":            # 무답변은 초안 생성 대상이 아니다
        continue
    n, want = made.get(kind, 0), TARGET[kind]
    mark = "✅" if n >= want * 2 else ("⚠️" if n >= want else "❌")
    print(f"  {mark} {kind:6s} 초안 {n:2d}개 / 목표 {want}개")
if failed:
    print(f"  생성 실패: {dict(Counter(k for k, _ in failed))}")

print("\n───────── 유형별 초안 내용 ─────────")
for kind in TARGET:
    if kind == "무답변":
        continue
    items = [(cs, d) for k, cs, d in drafts if k == kind]
    if not items:
        print(f"\n[{kind}] 초안 없음")
        continue
    print(f"\n[{kind}] {len(items)}개")
    for cs, d in items:
        src = cs[0].metadata.get("source", "?")[:22]
        pages = ",".join(str(c.metadata.get("page")) for c in cs)
        print(f"  · ({d.difficulty:6s}) {d.question[:70]}")
        print(f"      답: {str(d.answer)[:64]}")
        print(f"      근거: {src} p.{pages}")


[3/5] 초안 생성
   초안 61개 생성

───────── 유형별 초안 현황 ─────────
  ✅ 단일홉    초안 12개 / 목표 4개
  ✅ 멀티홉    초안  9개 / 목표 3개
  ✅ 표/절차   초안 15개 / 목표 3개
  ✅ 집계     초안  8개 / 목표 3개
  ✅ 조건     초안  5개 / 목표 2개
  ✅ 부정/예외  초안  6개 / 목표 2개
  ✅ 시점/기한  초안  6개 / 목표 2개

───────── 유형별 초안 내용 ─────────

[단일홉] 12개
  · (easy  ) 쇼라 모바일라이브의 대표 브랜딩은 무엇인가?
      답: ‘믿고사쇼라’
      근거: 1. 협력사 운영 통합 안내서 (p.23 p.3
  · (easy  ) 협력사가 세금계산서를 인증해야 하는 기간은 언제인가요?
      답: 매월 1~10일입니다.
      근거: 1. 협력사 운영 통합 안내서 (p.23 p.21
  · (easy  ) Hmall에서 유료 광고 집행을 원할 경우 누구와 협의해야 하나요?
      답: 담당 MD와 협의해야 합니다.
      근거: 2. Hmall 소개서 (p.30).pd p.24
  · (easy  ) 실크테라피 하이드로 피니셔 헤어 상품의 할인 후 판매가는 얼마인가요?
      답: 46,960원
      근거: 3. 쇼라 소개서 (p.22).pdf p.5
  · (easy  ) 현대홈쇼핑 카카오톡 플친 PUSH에 따르면 불앤체인 에코백 혜택은 언제 종료 임박이라고 안내되었나요?
      답: 9/20(수) 밤 10시
      근거: 3. 쇼라 소개서 (p.22).pdf p.17
  · (easy  ) 오른쪽 화면에서 '[건강&영양제] 닥터슈퍼칸 밀크씨슬 얼티브타민 12개월 外'의 가격은 얼마인가?
      답: 138,000원
      근거: 4. 광고 상품 소개서 (p.24).pd p.7
  · (easy  ) 슬라이딩배너 상품의 가격은 얼마인가?
      답: 120만원
   

In [17]:
# ---------- 4. 2단계 검수 ----------
print("\n[4/5] 검수")
# 공백을 모두 지운 문자열을 만든다.
# VLM 파싱 결과는 줄바꿈·공백이 원문과 미묘하게 달라서,
# 인용구를 그대로 비교하면 같은 문장인데도 불일치로 잡힌다.
squash = lambda t: re.sub(r"\s+", "", t)

# 탈락 항목을 (단계, 유형, 사유, 초안)으로 모아 마지막에 한꺼번에 보여준다.
# 어느 유형이 어느 단계에서 깎이는지 알아야 OVERSAMPLE이나 GUIDES를 고칠 수 있다.
rejected = []

# [1차] 인용 정합성 — 초안이 든 근거가 실제 원문에 있는지 글자 단위로 대조한다.
# 모델이 그럴듯하게 지어낸 문장을 여기서 걸러낸다.
stage1 = []
for kind, cs, d in drafts:
    # 근거가 2개(멀티홉)면 둘을 합쳐 놓고 그 안에 있는지 본다
    source = squash(" ".join(c.page_content for c in cs))
    # 8자 미만 인용은 우연히 일치할 수 있어 검증 대상에서 뺀다
    qs = [squash(q) for q in d.quotes if len(squash(q)) >= 8]
    if not qs:
        rejected.append(("1차 인용", kind, "인용구가 8자 미만이거나 비어 있음", d))
    elif not all(q in source for q in qs):
        bad = next(q for q in qs if q not in source)
        rejected.append(("1차 인용", kind, f"근거에 없는 인용: {bad[:40]}", d))
    else:
        stage1.append((kind, cs, d))
print(f"   1차 인용 정합성: 통과 {len(stage1)} / 탈락 {len(drafts) - len(stage1)}")

# 1.5차: 정답이 실질적 내용을 담고 있는지 (표의 빈 셀을 근거로 한 문항 제거)
# [1.5차] 정답 실질성 — 표의 빈 셀을 물어 정답이 '빈칸'이 되는 문항을 걸러낸다.
# 이런 문항은 맞혀도 검색 성능을 보여주지 못한다. 실제로 한 번 생성된 적이 있다.
EMPTY_ANSWER = re.compile(r"^\s*(빈칸|공란|없음|해당\s*없음|N/?A|-|비어\s*있(음|다))\s*\.?\s*$", re.I)

def has_substance(answer: str) -> bool:
    """표의 빈 셀을 물어 '빈칸'이 정답이 되는 문항은 검색 성능을 측정하지 못하므로 제외."""
    return bool(answer and not EMPTY_ANSWER.match(answer.strip()) and len(answer.strip()) >= 2)

before = len(stage1)
kept = []
for k, cs, d in stage1:
    if has_substance(d.answer):
        kept.append((k, cs, d))
    else:
        rejected.append(("1.5차 정답", k, f"정답이 실질적이지 않음: {str(d.answer)[:30]!r}", d))
stage1 = kept
print(f"   1.5차 정답 실질성: 통과 {len(stage1)} / 탈락 {before - len(stage1)}")


class SelfContained(BaseModel):
    ok: bool = Field(description="질문만 단독으로 읽고 무엇을 묻는지 특정되면 True")
    reason: str

# [2차] 자기완결성 — 질문만 단독으로 읽어도 무엇을 묻는지 특정되는지 LLM이 판정한다.
# "해당 표의 3번 항목은?" 처럼 문맥 없이는 답할 수 없는 질문을 걸러낸다.
# temperature=0 : 같은 질문에 매번 같은 판정이 나오도록 고정한다.
checker = ChatPromptTemplate.from_messages([
    ("system", "질문의 자기완결성을 평가합니다. 질문만 단독으로 전달했을 때 의도가 특정되는지 판단하세요. "
               "'이 문서', '해당 절차' 같은 모호한 표현이 있거나 주어가 빠졌으면 False입니다."),
    ("human", "질문: {question}"),
]) | ChatOpenAI(model=JUDGE, temperature=0).with_structured_output(SelfContained)

stage2 = []
for k, cs, d in stage1:
    v = checker.invoke({"question": d.question})
    if v.ok:
        stage2.append((k, cs, d))
    else:
        rejected.append(("2차 자기완결", k, v.reason[:60], d))
print(f"   2차 자기완결성: 통과 {len(stage2)} / 탈락 {len(stage1) - len(stage2)}")

# ---------- 5. 무답변 문항 자동 검증 ----------
# 1단계 파싱으로 회수된 텍스트가 3.8배 늘었으므로, 예전 무답변 후보가
# 이제는 문서에서 답이 나올 수 있다. 검색 + 심사로 '정말 답이 없는지' 확인한다.
print("\n[5/5] 무답변 후보 검증")

class Unanswerable(BaseModel):
    unanswerable: bool = Field(description="아래 문서 조각들만으로 질문에 답할 수 없으면 True")
    reason: str

# 무답변 후보를 검증할 심사 체인.
# 후보 질문으로 실제 검색을 돌린 뒤, 그 결과만으로 답할 수 있는지 판정한다.
# 1단계 파싱으로 회수 텍스트가 크게 늘어, 예전에 '문서에 없던' 질문이
# 이제는 답이 나올 수 있기 때문에 하드코딩하지 않고 매번 확인한다.
verifier = ChatPromptTemplate.from_messages([
    ("system", "주어진 문서 조각만으로 질문에 답할 수 있는지 판정합니다. 부분적으로라도 답이 되면 False입니다."),
    ("human", "질문: {q}\n\n문서 조각:\n{ctx}"),
]) | ChatOpenAI(model=JUDGE, temperature=0).with_structured_output(Unanswerable)

NEGATIVE_CANDIDATES = [
    "Hmall 협력사 담당 MD의 직통 전화번호는 몇 번인가?",
    "현대백화점 오프라인 매장 입점 신청 절차와 제출 서류는 무엇인가?",
    "Hmall의 2025년 연간 총 거래액 목표는 얼마인가?",
    "쇼라 방송 출연 쇼호스트의 회당 출연료는 얼마인가?",
    "Hmall 협력사 시스템의 서버 장애 시 비상 연락망은 무엇인가?",
    "Hmall에는 총 몇 개의 오프라인 매장이 존재하는가?",
]

negatives = []
for q in NEGATIVE_CANDIDATES:
    # 실제 검색기와 같은 방식으로 상위 5개 청크를 가져온다.
    # 이 청크들로 답이 안 나와야 진짜 '무답변' 문항이다.
    ctx = "\n---\n".join(d.page_content for d in gold_store.similarity_search(q, k=5))
    v = verifier.invoke({"q": q, "ctx": ctx})
    mark = "✅ 무답변 확정" if v.unanswerable else "❌ 답변 가능 → 제외"
    print(f"   {mark}: {q}")
    if v.unanswerable:
        negatives.append({
            "question": q,
            "ground_truth": "제공된 문서에서 정보를 찾을 수 없습니다.",
            "quote": "N/A",
            "question_type": "무답변",
            "source_context": "문서 내 해당 정보 없음",
        })
    if len(negatives) >= TARGET["무답변"]:
        break

# ---------- 6. 최종 구성 및 저장 ----------
# 같은 표가 여러 페이지에 반복되면 동일한 질문이 중복 생성된다.
# 중복 문항은 특정 표에 평가 가중치를 몰아주므로 제거한다.
# 검수를 통과한 초안 중에서 유형별 목표 개수만큼 앞에서부터 채운다.
# seen 은 이미 뽑힌 질문의 공백 제거 문자열 집합으로, 중복 문항을 막는다.
final, seen = [], set()
for kind in ["단일홉", "멀티홉", "표/절차", "집계", "조건", "부정/예외", "시점/기한"]:
    picked = []
    for x in stage2:
        if x[0] != kind or len(picked) >= TARGET[kind]:
            continue
        key = squash(x[2].question)
        if key in seen:
            rejected.append(("3차 중복", kind, "이미 선택된 문항과 질문이 동일", x[2]))
            continue
        seen.add(key)
        picked.append(x)
    if len(picked) < TARGET[kind]:
        print(f"   ⚠️ {kind}: 목표 {TARGET[kind]}개 중 {len(picked)}개만 확보")
    for _, cs, d in picked:
        final.append({
            "question": d.question,
            "ground_truth": d.answer,
            "quotes": d.quotes,
            "question_type": kind,
            "difficulty": d.difficulty,
            "source": cs[0].metadata.get("source", "?"),
            "page": cs[0].metadata.get("page"),
            "source_context": "\n---\n".join(c.page_content for c in cs),
        })
final.extend(negatives)

# ---------- 검수 탈락 리포트 ----------
import pandas as pd
from collections import Counter

STAGES = ["1차 인용", "1.5차 정답", "2차 자기완결", "3차 중복"]
KINDS = [k for k in TARGET if k != "무답변"]

rows = []
for kind in KINDS:
    row = {"유형": kind, "초안": sum(1 for k, _, _ in drafts if k == kind)}
    for st in STAGES:
        row[st] = sum(1 for s_, k, _, _ in rejected if k == kind and s_ == st)
    row["최종"] = sum(1 for x in final if x["question_type"] == kind)
    row["목표"] = TARGET[kind]
    rows.append(row)

print("\n───────── 검수 탈락 요약 (유형 × 단계) ─────────")
display(pd.DataFrame(rows).set_index("유형"))

print("───────── 탈락 문항 상세 ─────────")
for st in STAGES:
    items = [(k, r, d) for s_, k, r, d in rejected if s_ == st]
    if not items:
        continue
    print(f"\n[{st}] {len(items)}건  " + str(dict(Counter(k for k, _, _ in items))))
    for kind, reason, d in items:
        print(f"  · ({kind}) {d.question[:62]}")
        print(f"      사유: {reason}")

with open(GOLDEN_PATH, "w", encoding="utf-8") as f:
    json.dump(final, f, ensure_ascii=False, indent=4)

from collections import Counter
print(f"\n🎉 골든 데이터셋 {len(final)}문항 저장 → {GOLDEN_PATH}")
print("   ", Counter(x["question_type"] for x in final))


[4/5] 검수
   1차 인용 정합성: 통과 61 / 탈락 0
   1.5차 정답 실질성: 통과 61 / 탈락 0
   2차 자기완결성: 통과 27 / 탈락 34

[5/5] 무답변 후보 검증
   ❌ 답변 가능 → 제외: Hmall 협력사 담당 MD의 직통 전화번호는 몇 번인가?
   ✅ 무답변 확정: 현대백화점 오프라인 매장 입점 신청 절차와 제출 서류는 무엇인가?
   ✅ 무답변 확정: Hmall의 2025년 연간 총 거래액 목표는 얼마인가?
   ✅ 무답변 확정: 쇼라 방송 출연 쇼호스트의 회당 출연료는 얼마인가?
   ⚠️ 집계: 목표 3개 중 2개만 확보

───────── 검수 탈락 요약 (유형 × 단계) ─────────


,초안,1차 인용,1.5차 정답,2차 자기완결,3차 중복,최종,목표
유형,,,,,,,
단일홉,12,0,0,4,0,4,4
멀티홉,9,0,0,5,0,3,3
표/절차,15,0,0,11,0,3,3
집계,8,0,0,6,0,2,3
조건,5,0,0,3,0,2,2
부정/예외,6,0,0,3,0,2,2
시점/기한,6,0,0,2,0,2,2


───────── 탈락 문항 상세 ─────────

[2차 자기완결] 34건  {'단일홉': 4, '멀티홉': 5, '표/절차': 11, '집계': 6, '조건': 3, '부정/예외': 3, '시점/기한': 2}
  · (단일홉) 오른쪽 화면에서 '[건강&영양제] 닥터슈퍼칸 밀크씨슬 얼티브타민 12개월 外'의 가격은 얼마인가?
      사유: 질문이 '오른쪽 화면'이라는 외부 시각 자료를 참조하고 있어, 질문만으로는 해당 상품의 가격을 확인할 수 없
  · (단일홉) 상품의 배송 지연에 관한 고지문의 확인여부는 무엇인가?
      사유: 어떤 상품이나 어떤 배송 지연 고지문을 가리키는지 특정되지 않아, 질문만으로 확인여부를 판단할 대상이 불명확
  · (단일홉) 한글표시사항 또는 디자인 시안을 진행할 때 반드시 먼저 협의해야 하는 담당자는 누구인가?
      사유: 질문이 특정 문서나 조직/절차의 맥락을 전제로 하며, '반드시 먼저 협의해야 하는 담당자'가 누구인지는 질문
  · (단일홉) 색조화장품에 대해 요구되는 과불화화합물 관련 자료는 무엇인가?
      사유: 질문은 색조화장품과 과불화화합물 관련 자료를 묻고 있지만, 어떤 법령·기관·인증·신청 절차에서 요구되는 자료
  · (멀티홉) 완도 활전복 방송 글에서 제시된 할인 조건을 적용했을 때의 판매가 표현과, 쇼라 5주년 특집 방송 화면에 표시된
      사유: 질문이 특정 방송 글과 방송 화면의 내용을 전제로 하며, 해당 글/화면이 함께 제공되지 않으면 할인 조건, 
  · (멀티홉) 현대백화점탭 메인에서 화면 캡처 내 문구로 표시된 두 영역인 A_상단기획전과 B_지금 주목해야 할 브랜드의 1일
      사유: 질문이 특정 화면 캡처나 자료에 표시된 A_상단기획전, B_지금 주목해야 할 브랜드의 1일 비용과 노출 개수
  · (멀티홉) 협력사 온라인 문의를 접수할 때 함께 기입해야 하는 협력사 정보와, 출고지/회수지 현황에 등록된 담당자의 이름은
      사유: 질문이 특정 문서나 데이터

In [18]:
# =====================================================================
# 최종 골든 데이터셋 확인 — 유형별 출력
# ---------------------------------------------------------------------
# 저장된 golden_dataset.json 을 다시 읽어서 본다.
# 앞 셀들을 새로 실행하지 않아도 이 셀만으로 결과를 확인할 수 있다.
# =====================================================================
import json
from pathlib import Path
from collections import Counter

import pandas as pd

if not Path(GOLDEN_PATH).exists():
    print(f"⚠️ {GOLDEN_PATH} 가 없습니다. 앞의 골든셋 생성 셀을 먼저 실행하세요.")
else:
    # 메모리의 final 변수가 아니라 저장된 파일을 읽는다.
    # 실제로 파일에 무엇이 들어갔는지 확인하는 것이 목적이고,
    # 앞 셀을 다시 실행하지 않아도 이 셀만으로 결과를 볼 수 있다.
    gold = json.loads(Path(GOLDEN_PATH).read_text(encoding="utf-8"))

    # ---------- 구성 요약 ----------
    made = Counter(g["question_type"] for g in gold)
    summary = pd.DataFrame([
        {"유형": k, "목표": TARGET[k], "생성": made.get(k, 0),
         "충족": "○" if made.get(k, 0) >= TARGET[k] else f"✗ {TARGET[k]-made.get(k,0)}개 부족"}
        for k in TARGET
    ]).set_index("유형")
    print(f"===== 골든 데이터셋 {len(gold)}문항 / 목표 {sum(TARGET.values())}문항 =====")
    display(summary)

    # 난이도·출처 분포 (무답변은 근거 문서가 없으므로 제외)
    # 무답변 문항은 근거 문서가 없으므로 난이도·출처 집계에서 뺀다.
    answerable = [g for g in gold if g["question_type"] != "무답변"]
    if answerable:
        print("난이도:", dict(Counter(g.get("difficulty", "-") for g in answerable)))
        print("출처 문서:", dict(Counter(g.get("source", "?")[:18] for g in answerable)))

    # ---------- 유형별 문항 ----------
    for kind in TARGET:
        items = [g for g in gold if g["question_type"] == kind]
        print("\n" + "=" * 100)
        print(f"[{kind}] {len(items)}문항")
        print("=" * 100)
        if not items:
            print("  (없음)")
            continue
        for i, g in enumerate(items, 1):
            print(f"\n{i}. {g['question']}")
            print(f"   정답: {g['ground_truth']}")
            if kind == "무답변":
                continue
            print(f"   난이도: {g.get('difficulty','-')}   "
                  f"근거: {g.get('source','?')} p.{g.get('page','-')}")
            for q in g.get("quotes", []):
                print(f"   인용: {q[:90]}")

===== 골든 데이터셋 21문항 / 목표 22문항 =====


,목표,생성,충족
유형,,,
단일홉,4,4,○
멀티홉,3,3,○
표/절차,3,3,○
집계,3,2,✗ 1개 부족
조건,2,2,○
부정/예외,2,2,○
시점/기한,2,2,○
무답변,3,3,○


난이도: {'easy': 16, 'medium': 2}
출처 문서: {'1. 협력사 운영 통합 안내서 (': 3, '2. Hmall 소개서 (p.30': 5, '3. 쇼라 소개서 (p.22).p': 2, '4. 광고 상품 소개서 (p.24': 3, '6. 신규 협력사 입점 절차 안내': 1, '8. QA가이드 (p.38).pd': 4}

[단일홉] 4문항

1. 쇼라 모바일라이브의 대표 브랜딩은 무엇인가?
   정답: ‘믿고사쇼라’
   난이도: easy   근거: 1. 협력사 운영 통합 안내서 (p.23).pdf p.3
   인용: · 대표브랜딩‘믿고사쇼라’

2. 협력사가 세금계산서를 인증해야 하는 기간은 언제인가요?
   정답: 매월 1~10일입니다.
   난이도: easy   근거: 1. 협력사 운영 통합 안내서 (p.23).pdf p.21
   인용: **‘매월1~10일’세금계산서인증필요**

3. Hmall에서 유료 광고 집행을 원할 경우 누구와 협의해야 하나요?
   정답: 담당 MD와 협의해야 합니다.
   난이도: easy   근거: 2. Hmall 소개서 (p.30).pdf p.24
   인용: 광고 집행을 원하실 경우 담당 MD와 협의해주세요.

4. 실크테라피 하이드로 피니셔 헤어 상품의 할인 후 판매가는 얼마인가요?
   정답: 46,960원
   난이도: easy   근거: 3. 쇼라 소개서 (p.22).pdf p.5
   인용: [2024최신상] 실크테라피 하이드로 피니셔 헤어 ...
   인용: 10% 46,960원 52,170원

[멀티홉] 3문항

1. 현대홈쇼핑이 보유한 판매 채널 중 MZ 고객층과의 가교 역할을 하며 대표 브랜딩이 ‘믿고사쇼라’인 채널은 회사 개요에서 어떤 분류의 채널로 제시되나요?
   정답: 쇼라이며, 회사 개요에서는 라이브커머스 채널로 제시됩니다.
   난이도: medium   근거: 1. 협력사 운영 통합 안내서 (p.23).pdf p.3
   인용: · MZ 고객층과의가교역할
   인용: 라이브커머스